# Honeypot log analysis

Rebuilds the numbers and charts in [README_REPORT.md](../README_REPORT.md) from the visit logs
that `deploy/daily-logs.sh` copies into `~/honeypot-logs`. Run the cells from the top. Sections 7
to 10 go further than the report: campaigns, the devices the logins were made for, where the IPs
are registered, and whether anyone had reported them already. The check near the end compares
every number with the report.

**Handling the logs safely**

- **IPs:** the loader turns each address into a masked label (first two numbers) and an ID made
  with a random key that only exists while the kernel runs. IPs can still be counted and matched
  across periods, but no full address is ever kept or shown. Country and ASN come from MaxMind
  files on this computer, looked up inside `load_geo`, so no IP is sent anywhere for them.
- **Passwords:** read once, only to find my own test visits (password `not-on-the-list`), then
  dropped.
- **Commands** appear only in the playbook and campaign tables: defanged (`hxxp`, `[.]`),
  shortened, and never as links. The campaign table names long lines, such as downloads, by
  their step instead. **Usernames** appear only when several IPs tried them, as in the daily
  summary, because a one-off could be a password typed into the wrong box. Section 8 also
  replaces any IPv4 address inside a username with `<IP>`, since one scanner sends the
  honeypot's own address.
- **Nothing from the logs is ever fetched, opened or run.** The only thing that contacts anyone
  is `analysis/threat_intel_lookup.py`, which you run by hand: it sends IPs to AbuseIPDB's check
  lookup and download addresses to URLhaus's lookups, and never submits anything. This notebook
  only reads its results, turning IPs and addresses into opaque IDs as it reads them.
- **Git:** outputs hold log data. `.gitattributes` strips them from this notebook on every commit
  and `analysis/figures/` is ignored, but check `git diff --cached` before committing anyway. On a
  fresh clone, set the filter up first:
  `git config filter.strip-notebook-output.clean "jq --indent 1 -f analysis/strip-outputs.jq"`
  and `git config filter.strip-notebook-output.required true`.
- Never display `visits_all`, `visits`, `after`, `sessions` or `cmds` directly: they still hold
  commands and usernames.

In [ ]:
import hashlib
import hmac
import html
import json
import re
import secrets
from datetime import datetime, timedelta, timezone
from pathlib import Path
from urllib.parse import urlsplit

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

pd.set_option("display.max_colwidth", 100)

# The repository root, found from wherever the notebook runs.
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Cargo.toml").exists())
LOG_DIR = Path.home() / "honeypot-logs"


def utc(text):
    return datetime.fromisoformat(text).replace(tzinfo=timezone.utc)


# From the deployment record. All times are UTC.
STAGE6_OPEN = utc("2026-11-02 00:17:40")  # fake login only, on the old server
PORT_CLOSED = utc("2026-11-03 01:00:16")  # closed for the reinstall
CHANGE = utc("2026-11-03 01:29:49")  # reopened with the telnet opening and shell, on the same IP
# The report covers the shell from its launch to this visit, the last in the logs copied on Nov 10.
REPORT_END = utc("2026-11-10 19:45:02")
# The telnet-opening comparison (section 3) keeps its own window: the first study, Nov 3 to Nov 6,
# ending at that study's last visit.
TELNET_END = utc("2026-11-06 15:34:11")
WINDOW = "Nov 3–10"  # the report's window, for chart subtitles

# The report's "before" column is the daily summary run at 00:59 on Nov 3: the 24 hours before it.
BASELINE_END = utc("2026-11-03 00:59:00")
BASELINE_START = BASELINE_END - timedelta(hours=24)

# Set END = None to include logs copied since the report. The checks at the bottom then won't match.
END = REPORT_END

# "light" for the report and site images, "dark" to match a dark editor.
THEME = "light"

## Load the logs

Full IPs and passwords only exist inside `load_visits`. What comes out has a masked label and an
opaque ID for each IP, and no passwords.

In [ ]:
# A new random key each time the kernel starts. Without it, an ID can't be turned back into an
# address, even by trying every IPv4 address.
_KEY = secrets.token_bytes(32)


def opaque_id(text):
    return hmac.new(_KEY, text.encode(), hashlib.sha256).hexdigest()[:16]


def mask_ip(ip):
    parts = ip.split(".")
    return f"{parts[0]}.{parts[1]}.x.x" if len(parts) == 4 else "(not IPv4)"


def url_port(url):
    """The port a download address names, read from its text. Nothing is contacted."""
    if "://" not in url:
        return None  # a bare host, as tftp takes it
    try:
        parts = urlsplit(url)
        return parts.port or {"http": 80, "https": 443}.get(parts.scheme.lower())
    except ValueError:
        return None


def url_host(url):
    """The server an address names, read from its text. Nothing is contacted."""
    if "://" not in url:
        return url  # a bare host, as tftp takes it
    try:
        return urlsplit(url).hostname or url
    except ValueError:
        return url


def mask_address(text):
    """An address with every IPv4 address in it cut to its first two numbers, like the IPs."""
    return re.sub(r"(?<!\d)(\d{1,3})\.(\d{1,3})\.\d{1,3}\.\d{1,3}(?!\d)", r"\1.\2.x.x", text)


def load_visits(log_dir):
    files = sorted(log_dir.glob("visits-*.jsonl"))
    raw, skipped = [], 0
    for path in files:
        with path.open(encoding="utf-8", errors="replace") as f:
            for line in f:
                try:
                    raw.append(json.loads(line))
                except json.JSONDecodeError:
                    skipped += 1  # a line cut off while the log was being copied
    # The one use of passwords: my own test visits used this one, so their IPs are left out.
    own = {v["source_ip"] for v in raw if v.get("password") == "not-on-the-list"}
    rows = []
    for v in raw:
        ip = v["source_ip"]
        shell = v.get("shell") or {}
        urls = shell.get("urls") or []
        rows.append(
            {
                "t": v["time"],
                "time": pd.Timestamp(v["time"], unit="s", tz="UTC"),
                "ip_id": opaque_id(ip),
                "ip_masked": mask_ip(ip),
                "excluded": ip.startswith("127.") or ip in own,
                "outcome": v["outcome"],
                "username": v.get("username"),
                "telnet_answered": bool(v.get("telnet")),
                "ended": shell.get("ended"),
                "commands": shell.get("commands") or [],
                "url_ids": [opaque_id(u) for u in urls],
                "url_ports": [url_port(u) for u in urls],
                "url_hosts": [opaque_id(url_host(u)) for u in urls],
                "urls_masked": [mask_address(u) for u in urls],
            }
        )
    print(
        f"Read {len(raw):,} lines from {len(files)} files, skipped {skipped} unreadable ones, "
        f"and found {len(own)} of my own test IPs to leave out."
    )
    return pd.DataFrame(rows)


visits_all = load_visits(LOG_DIR)
assert not {"source_ip", "password", "telnet"} & set(visits_all.columns)
visits = visits_all[~visits_all["excluded"]]

## Periods and shared rules

The same rules as the daily summary and the report: a visit is silent when it sent no username
line, and a retry-pattern IP has 5 or more silent visits about 13 seconds apart.

In [ ]:
in_range = visits["time"] <= END if END else pd.Series(True, index=visits.index)
after = visits[(visits["time"] >= CHANGE) & in_range]
before = visits[(visits["time"] >= BASELINE_START) & (visits["time"] < BASELINE_END)]
first_day = after[after["time"] < CHANGE + timedelta(hours=24)]
sessions = after[after["outcome"] == "logged_in"].copy()
after_telnet = after[after["time"] <= TELNET_END]  # the telnet comparison's "after", Nov 3 to Nov 6


def silent(df):
    """Visits that sent nothing at all."""
    return df[df["username"].isna()]


def sent_login(df):
    return df[df["outcome"].isin(["attempted", "logged_in"])]


def pct(part, whole, digits=0):
    value = round(100 * part / whole, digits)
    return int(value) if digits == 0 else value


def retry_ips(df):
    """IPs with 5+ silent visits whose median gap is 12 to 15 seconds."""
    found = set()
    for ip, times in silent(df).groupby("ip_id")["t"]:
        t = sorted(times)
        if len(t) < 5:
            continue
        gaps = sorted(b - a for a, b in zip(t, t[1:]))
        if 12 <= gaps[len(gaps) // 2] <= 15:
            found.add(ip)
    return found


def printable(text):
    return "".join(ch for ch in text if " " <= ch <= "~")


def safe_usernames(df, min_ips=2):
    """Usernames tried by at least min_ips IPs, with control characters removed."""
    named = df[df["username"].fillna("") != ""]
    table = named.groupby("username").agg(attempts=("ip_id", "size"), ips=("ip_id", "nunique"))
    table = table[table["ips"] >= min_ips].sort_values(["ips", "attempts"], ascending=False)
    table.index = [printable(u)[:32] for u in table.index]
    table.index.name = "username"
    return table


hours_live = (after["time"].max() - CHANGE).total_seconds() / 3600
print(
    f"Before: {len(before):,} visits in the 24 hours to {BASELINE_END:%b %d %H:%M} UTC. "
    f"After: {len(after):,} visits over {hours_live:.0f} hours, {len(sessions)} of them shell sessions."
)

## Sort commands into playbook steps

Each command line gets one family, checked in a fixed order. A line that holds several commands
also counts for any check or look-around another of them runs, as in `ls /home; /bin/busybox
BOTNET`. Only those: split into commands, `ping ;sh` would stop being a ping injection. Removing
rivals and cleaning up are found inside download lines, so they're extra flags.

In [ ]:
MENU_ESCAPE = {"enable", "system", "shell", "sh", "linuxshell", "enablelinuxshell"}
# Some bots end these words with a NUL byte, as in Mirai's string tables. The log stores it as \0.
MENU_ESCAPE |= {w + "\\0" for w in ["enable", "system", "shell", "sh", "linuxshell"]}
ROUTER_CLI = {"config terminal", "start", "su"}
LOOK_AROUND = {"cat /proc", "uname -m", "cat /etc/passwd", "id", "busybox", "/bin/busybox"}
# The checks only a real shell passes. The writable-folder probe counts only in its form that ends
# with a hex echo marker (is_marker_probe), not the plainer forms seen after Nov 6.
CHECKS = {"busybox_tag", "echo_marker", "shell_test"}
# A file made in a folder, then cd into it: ">/tmp/.ptmx && cd /tmp/", or BusyBox writing an empty
# script there, running it and cd-ing in. The marker probe starts the same way.
WRITABLE = re.compile(r"(?:/bin/busybox echo )?>\s*/[^\s;&]*\s*&&\s*(?:sh /[^\s;&]*\s*&&\s*)?cd /")


def is_marker_probe(cmd):
    return cmd.startswith(">/var/run/.x&&")


def family(cmd):
    if cmd == "echo SHELL_TEST":
        return "shell_test"
    if is_marker_probe(cmd) or WRITABLE.match(cmd):
        return "writable_probe"
    if re.search(r"wget|curl|tftp|ftpget", cmd):
        return "download"
    # A made-up applet holds a capital letter. Real applets, such as echo or hostname, are lower case.
    if re.match(r"/bin/busybox [a-z0-9]*[A-Z]", cmd):
        return "busybox_tag"
    if cmd.startswith("echo -e") or cmd == "echo PAYLOAD_WORKING":
        return "echo_marker"
    if cmd in MENU_ESCAPE:
        return "menu_escape"
    if cmd in ROUTER_CLI:
        return "router_cli"
    if re.fullmatch(r"ping ?; ?sh", cmd):
        return "ping_injection"
    if cmd in LOOK_AROUND or re.match(r"cat /proc/mounts|ls /home", cmd):
        return "look_around"
    return "other"


# The steps a line also counts for when another of its commands takes them.
ALSO_COUNTED = CHECKS | {"look_around"}


def line_steps(cmd):
    """The line's own family, plus any check or look-around in its other commands."""
    parts = re.split(r"\s*(?:;|&&|\|\|)\s*", cmd)
    return {family(cmd)} | ({family(part) for part in parts} & ALSO_COUNTED)


RIVALS = re.compile(r"for p in ([^;]*);\s*do\s+killall -9 \$p")
# Run a script, then delete it: "sh x.sh; rm -f x.sh", or "sh ./x.sh;rm -rf x.sh".
CLEAN_UP = re.compile(r"\bsh (?:\./)?(\S+);\s*rm -r?f (?:\./)?\1\b")
# Hex-escaped echo -ne lines that write a script, as logged: an escaped backslash, x, two hex digits.
HEX_BYTE = re.compile(r"\\\\x([0-9a-fA-F]{2})")
KILL_PATTERN = re.compile(r"\*([A-Za-z0-9]+)\*")  # a name in a shell case pattern, such as *mirai*


def hex_script(commands):
    """What a session wrote with hex-escaped echo -ne lines, decoded as text. Nothing is run."""
    return "".join(chr(int(h, 16)) for c in commands if "echo -ne" in c for h in HEX_BYTE.findall(c))


def kills_rivals(script):
    return "kill -9" in script and len(KILL_PATTERN.findall(script)) >= 5

cmds = sessions[["ip_id", "commands"]].explode("commands").dropna(subset=["commands"])
cmds = cmds.rename(columns={"commands": "cmd"})
cmds["session"] = cmds.index
cmds["family"] = cmds["cmd"].map(family)
cmds["steps"] = cmds["cmd"].map(line_steps)
# A rival-killing script written with hex echoes counts once, on the line that starts it.
rival_scripts = set(sessions.index[sessions["commands"].map(lambda cs: kills_rivals(hex_script(cs)))])
is_hex_echo = cmds["cmd"].str.contains("echo -ne", regex=False).to_numpy()
first_hex_echo = is_hex_echo & (pd.Series(is_hex_echo).groupby(cmds["session"].to_numpy()).cumsum().to_numpy() == 1)
cmds["remove_rivals"] = (cmds["cmd"].map(lambda c: RIVALS.search(c) is not None).to_numpy()
                         | (first_hex_echo & cmds["session"].isin(rival_scripts).to_numpy()))
cmds["clean_up"] = cmds["cmd"].map(lambda c: CLEAN_UP.search(c) is not None)

families = cmds.groupby("session")["steps"].agg(lambda steps: set().union(*steps)).reindex(sessions.index)
sessions["families"] = families.map(lambda f: f if isinstance(f, set) else set())
sessions["ran_check"] = (sessions["families"].map(lambda f: bool(f & CHECKS))
                         | sessions["commands"].map(lambda cs: any(is_marker_probe(c) for c in cs)))
sessions["downloaded"] = sessions["url_ids"].map(len) > 0
print(f"{len(cmds):,} commands, {cmds['cmd'].nunique()} different, in {cmds['session'].nunique()} sessions.")

## Defanging, for the one table that shows commands

In [ ]:
# Digit lookarounds, not \b, so an address straight after a letter or underscore is caught too.
IPV4 = re.compile(r"(?<!\d)(\d{1,3})\.(\d{1,3})\.\d{1,3}\.\d{1,3}(?!\d)")
# Long letter-and-digit path parts look like campaign IDs, which could identify this honeypot.
CAMPAIGN_ID = re.compile(r"(?<=/)(?=[A-Za-z0-9]*\d)(?=[A-Za-z0-9]*[A-Za-z])[A-Za-z0-9]{8,}(?=/)")


def defang(text, limit=90):
    r"""Make a logged command safe to show: no full IPs, no working links, no control characters.

    The log stores commands escaped (Rust's escape_debug): backslashes doubled, quotes escaped,
    and control characters written as \0, \t or \u{...}. Undoing the first two shows the line as
    the bot sent it, while control characters stay visible as text.
    """
    text = re.sub(r"\\([\\'\"])", r"\1", text)
    text = printable(text)
    text = IPV4.sub(lambda m: f"{m[1]}[.]{m[2]}[.]x[.]x", text)
    text = re.sub(r"(?i)\bhttp", "hxxp", text)
    text = re.sub(r"(hxxps?://)([A-Za-z0-9.-]+)", lambda m: m[1] + m[2].replace(".", "[.]"), text)
    text = CAMPAIGN_ID.sub("<id>", text)
    return text if len(text) <= limit else text[: limit - 1] + "…"


def example_for(key, lines):
    common = lines.value_counts().index[0]
    if key == "remove_rivals":
        killall = next((m[0] for m in map(RIVALS.search, lines) if m), None)
        return defang(killall) if killall else "(a hex-echoed script)"
    if key == "clean_up":
        return defang(re.sub(r"\S+\.sh", "<name>.sh", CLEAN_UP.search(common)[0]))
    return defang(common)

## Chart style

Thin marks, one hue for a single series, a validated blue and orange pair when two series are
compared, and hairline grids. Each chart is followed by its numbers as a table.

In [ ]:
TOKENS = {
    "light": {"surface": "#fcfcfb", "ink": "#0b0b0b", "ink2": "#52514e", "muted": "#898781",
              "grid": "#e1e0d9", "axis": "#c3c2b7", "s1": "#2a78d6", "s2": "#eb6834",
              "wash": "rgba(11,11,11,0.06)"},
    "dark": {"surface": "#1a1a19", "ink": "#f0efec", "ink2": "#c3c2b7", "muted": "#898781",
             "grid": "#2c2c2a", "axis": "#383835", "s1": "#3987e5", "s2": "#d95926",
             "wash": "rgba(255,255,255,0.08)"},
}
T = TOKENS[THEME]
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'
FIGURES = {}


def style(fig, title, subtitle=None, height=420):
    title_spec = {"text": title, "x": 0.01, "xanchor": "left", "font": {"size": 18, "color": T["ink"]}}
    if subtitle:
        title_spec["subtitle"] = {"text": subtitle, "font": {"size": 13, "color": T["ink2"]}}
    axis = {"gridcolor": T["grid"], "gridwidth": 1, "linecolor": T["axis"], "zeroline": False,
            "tickfont": {"color": T["muted"]}, "title": {"font": {"color": T["ink2"]}}}
    fig.update_layout(
        title=title_spec,
        height=height,
        paper_bgcolor=T["surface"],
        plot_bgcolor=T["surface"],
        font={"family": FONT, "color": T["ink"], "size": 13},
        margin={"l": 24, "r": 48, "t": 110 if subtitle else 70, "b": 48},
        hoverlabel={"bgcolor": T["surface"], "bordercolor": T["axis"],
                    "font": {"color": T["ink"], "family": FONT}},
        legend={"orientation": "h", "x": 0, "y": 1.0, "yanchor": "bottom", "font": {"color": T["ink2"]}},
        barcornerradius=4,
    )
    fig.update_xaxes(**axis)
    fig.update_yaxes(**axis)
    return fig


def show(fig, name):
    FIGURES[name] = fig
    fig.show(config={"displaylogo": False,
                     "toImageButtonOptions": {"format": "png", "scale": 2, "filename": name}})


def naive(moment):
    """Plain UTC time for a chart axis."""
    return pd.Timestamp(moment).tz_convert(None)

## 1. From connection to download

In [ ]:
with_command = sessions["commands"].map(len) > 0
funnel = pd.DataFrame({
    "stage": ["Connections", "Sent a login", "Got into the shell", "Sent a command",
              "Ran a real-shell check", "Reached a download"],
    "count": [len(after), len(sent_login(after)), len(sessions), int(with_command.sum()),
              int(sessions["ran_check"].sum()), int(sessions["downloaded"].sum())],
})
funnel["share of connections"] = (100 * funnel["count"] / len(after)).round(1).astype(str) + "%"
checked_then_downloaded = int((sessions["ran_check"] & sessions["downloaded"]).sum())

fig = go.Figure(go.Funnel(
    y=funnel["stage"], x=funnel["count"], width=0.55, cliponaxis=False,
    texttemplate="%{value:,}  ·  %{percentInitial:.1%}", textposition="outside", textfont={"color": T["ink"]},
    marker={"color": T["s1"]},
    connector={"fillcolor": T["wash"], "line": {"width": 0}},
    hovertemplate="%{y}: %{x:,}, %{percentInitial:.1%} of connections<extra></extra>",
))
style(fig, "Most bots that got in left before downloading anything",
      f"{WINDOW}. The last two steps overlap rather than nest: {checked_then_downloaded} of the "
      f"{funnel['count'].iloc[-1]} sessions that reached a download ran a check first.")
fig.update_yaxes(showgrid=False, showline=False, ticks="")
fig.update_xaxes(visible=False)
fig.update_layout(margin={"r": 150})  # room for the labels beside the widest bars
show(fig, "1-funnel")
display(funnel)

## 2. Visits per hour, before and after the change

In [ ]:
span = visits[(visits["time"] >= STAGE6_OPEN) & in_range]
hourly = span.set_index("time").resample("1h").size().rename("visits")
hourly.index = hourly.index.tz_convert(None)

fig = go.Figure(go.Scatter(
    x=hourly.index, y=hourly.values, mode="lines", line={"color": T["s1"], "width": 2},
    hovertemplate="%{x|%b %d, %H:00} UTC: %{y} visits<extra></extra>",
))
fig.add_vrect(x0=naive(PORT_CLOSED), x1=naive(CHANGE), fillcolor=T["wash"], line_width=0, layer="below")
fig.add_shape(type="line", x0=naive(CHANGE), x1=naive(CHANGE), y0=0, y1=1, yref="paper",
              line={"color": T["ink2"], "width": 1})
for text, anchor in [("← Fake login only  ", "right"), ("  Telnet opening and fake shell →", "left")]:
    fig.add_annotation(x=naive(CHANGE), y=1, yref="paper", yanchor="bottom", xanchor=anchor,
                       text=text, showarrow=False, font={"color": T["ink2"], "size": 12})
style(fig, "Visits per hour, before and after the change",
      "UTC, from the fake login going live on Nov 2. The thin shaded strip is the 30 minutes the "
      "port was closed for the reinstall.")
fig.update_xaxes(showgrid=False, showline=True)
fig.update_yaxes(title_text="Visits per hour", rangemode="tozero")
show(fig, "2-visits-per-hour")

per_day = after.groupby(after["time"].dt.date).apply(
    lambda g: pd.Series({"visits": len(g), "silent": len(silent(g)),
                         "silent %": pct(len(silent(g)), len(g), 1), "shell sessions": int((g["outcome"] == "logged_in").sum())}),
    include_groups=False,
)
per_day.index.name = "UTC day, after the change"
display(per_day)

## 3. Silent visits before and after the change

The server kept the same IP address through the change, so the address isn't a confound. Visitor
turnover is: few IPs were seen in both periods (the third pair of bars), and the baseline is one
day.

In [ ]:
retry_before = retry_ips(before)
# The comparison keeps its own "after" window, Nov 3 to Nov 6 (after_telnet).
seen_both = set(before["ip_id"]) & set(after_telnet["ip_id"])
before_no_retry = before[~before["ip_id"].isin(retry_before)]
retry_after = after_telnet[after_telnet["ip_id"].isin(retry_before)]
b_both, a_both = before[before["ip_id"].isin(seen_both)], after_telnet[after_telnet["ip_id"].isin(seen_both)]

silent_table = pd.DataFrame({
    "group": ["All visits", f"Without the {len(retry_before)} retry IPs", f"The {len(seen_both)} IPs seen in both"],
    "before silent": [len(silent(before)), len(silent(before_no_retry)), len(silent(b_both))],
    "before visits": [len(before), len(before_no_retry), len(b_both)],
    "after silent": [len(silent(after_telnet)), len(silent(retry_after)) if len(retry_after) else None,
                     len(silent(a_both))],
    "after visits": [len(after_telnet), len(retry_after) if len(retry_after) else None, len(a_both)],
})
silent_table["before %"] = [pct(s, n) for s, n in zip(silent_table["before silent"], silent_table["before visits"])]
silent_table["after %"] = pd.array(
    [pct(s, n) if pd.notna(n) and n else None for s, n in zip(silent_table["after silent"], silent_table["after visits"])],
    dtype="Int64",
)

fig = go.Figure()
# Narrow bars side by side, 0.08 of each category wide with a small gap between them.
for period, color, label, offset in [("before", T["s1"], "Before: 24 hours, fake login only", -0.09),
                                     ("after", T["s2"], "After: Nov 3–6, telnet opening and shell", 0.01)]:
    values = [int(p) if pd.notna(p) else None for p in silent_table[f"{period} %"]]
    fig.add_bar(
        x=silent_table["group"], y=values, name=label, marker={"color": color}, width=0.08, offset=offset,
        text=[f"{p}%" if p is not None else "" for p in values],
        textposition="outside", textfont={"color": T["ink"]}, cliponaxis=False,
        customdata=silent_table[[f"{period} silent", f"{period} visits"]],
        hovertemplate=label + "<br>%{x}: %{y}% silent (%{customdata[0]} of %{customdata[1]} visits)<extra></extra>",
    )
if not len(retry_after):
    fig.add_annotation(x=silent_table["group"][1], y=1, xshift=3, text="never came back", showarrow=False,
                       xanchor="left", yanchor="bottom", font={"color": T["muted"], "size": 12})
b, a = silent_table["before %"][0], silent_table["after %"][0]
style(fig, f"Silent visits fell from {b}% to {a}% after the change",
      "The address didn't change, but the visitors mostly did, so the telnet opening isn't proven to be the cause.",
      height=440)
fig.update_yaxes(range=[0, 100], ticksuffix="%", showline=False)
fig.update_xaxes(showgrid=False, showline=True)
show(fig, "3-silent-before-after")
display(silent_table)

## 4. The post-login playbook, in attack order

The chart counts sessions. The table also gives command lines and IPs, which are the numbers in
the report's playbook table. Commands in it are defanged examples, the most common line in each
step.

In [ ]:
STEPS = [
    ("login", "Log in with factory defaults", "T1110.001, T1078.001"),
    ("menu_escape", "Break out of a router's limited menu", "T1059.008, T1059.004"),
    ("router_cli", "Try a router CLI", "T1059.008"),
    ("ping_injection", "Inject a shell into a ping tool", "T1059.004"),
    ("look_around", "Look around", "T1082, T1087.001, T1033, T1083"),
    ("writable_probe", "Find a writable folder, then check with a hex echo", "T1083"),
    ("busybox_tag", "Check for a real BusyBox", "T1497.001"),
    ("echo_marker", "Check that echo works", "T1497.001"),
    ("shell_test", "The SHELL_TEST check", "T1497.001"),
    ("download", "Download and run", "T1105, T1222.002, T1059.004"),
    ("remove_rivals", "Remove rival bots", "T1489"),
    ("clean_up", "Clean up", "T1070.004"),
]
usernames = safe_usernames(after)

rows = []
for key, step, attack in STEPS:
    if key == "login":
        lines, n_sessions, ips = len(sent_login(after)), len(sessions), sessions["ip_id"].nunique()
        example = "usernames such as " + ", ".join(usernames.index[:4])
    else:
        chosen = (cmds[cmds[key]] if key in ("remove_rivals", "clean_up")
                  else cmds[cmds["steps"].map(lambda steps: key in steps)])
        lines, n_sessions, ips = len(chosen), chosen["session"].nunique(), chosen["ip_id"].nunique()
        example = example_for(key, chosen["cmd"])
    rows.append({"key": key, "step": step, "ATT&CK": attack, "sessions": n_sessions,
                 "lines": lines, "IPs": ips, "example (defanged)": example})
playbook = pd.DataFrame(rows)

fig = go.Figure(go.Bar(
    x=playbook["sessions"], y=[f"{s}  ·  {a}" for s, a in zip(playbook["step"], playbook["ATT&CK"])],
    orientation="h", marker={"color": T["s1"]},
    text=playbook["sessions"], textposition="outside", textfont={"color": T["ink"]}, cliponaxis=False,
    customdata=playbook[["lines", "IPs"]],
    hovertemplate="%{y}<br>%{x} sessions, %{customdata[0]:,} lines, %{customdata[1]} IPs<extra></extra>",
))
style(fig, "What bots did after logging in, in attack order",
      f"Sessions that took each step, {WINDOW}, with MITRE ATT&CK IDs. Most sessions took several steps.",
      height=600)
fig.update_layout(bargap=0.4)
fig.update_yaxes(autorange="reversed", showgrid=False, ticks="", showline=False, tickfont={"color": T["ink2"]})
fig.update_xaxes(title_text="Sessions", showline=True)
show(fig, "4-playbook")
other = cmds[cmds["steps"].map(lambda steps: steps == {"other"})]
print(f"Not in a step: {len(other)} empty, non-text or garbled lines from {other['ip_id'].nunique()} IPs.")
display(playbook.drop(columns="key"))

## 5. Usernames tried

Ranked by how many IPs tried each name, so one bot repeating itself can't top the list.
Passwords are never shown.

In [ ]:
top_users = usernames.head(10)
fig = go.Figure(go.Bar(
    x=top_users["ips"], y=[html.escape(u) for u in top_users.index], orientation="h",
    marker={"color": T["s1"]}, text=top_users["ips"], textposition="outside",
    textfont={"color": T["ink"]}, cliponaxis=False, customdata=top_users[["attempts"]],
    hovertemplate="%{y}: tried by %{x} IPs, %{customdata[0]:,} attempts<extra></extra>",
))
style(fig, "The usernames bots tried most widely",
      f"{WINDOW}, top 10 by number of IPs. Names tried by only one IP are left out, because a "
      "one-off could be a password typed into the wrong box.", height=460)
fig.update_layout(bargap=0.4)
fig.update_yaxes(autorange="reversed", showgrid=False, ticks="", showline=False, tickfont={"color": T["ink2"]})
fig.update_xaxes(title_text="IP addresses that tried it", showline=True)
show(fig, "5-usernames")
display(top_users)

## 6. How shell sessions ended

In [ ]:
ENDED = {"left": "Hung up", "timed_out": "Hit the 30-second limit", "exit": "Typed exit",
         "over_limit": "Went over a session limit"}
ended = sessions["ended"].value_counts().rename(index=ENDED)
total = int(ended.sum())

fig = go.Figure()
for (label, n), color in zip(ended.items(), [T["s1"], T["s2"], T["muted"], T["axis"]]):
    share = n / total
    fig.add_bar(
        y=["sessions"], x=[n], orientation="h", name=label, width=0.3,
        marker={"color": color, "line": {"color": T["surface"], "width": 2}},
        text=[f"  {n:,} ({share:.0%})"], textposition="inside", insidetextanchor="start",
        textfont={"color": "#ffffff"},
        hovertemplate=f"{label}: {n} sessions ({share:.0%})<extra></extra>",
    )
note = "None typed exit. " if "Typed exit" not in ended.index else ""
style(fig, f"How the {total} shell sessions ended",
      f"{WINDOW}. {note}Every visit has a 30-second limit, so a session that hit it held the bot for the full 30 seconds.",
      height=230)
fig.update_layout(barmode="stack", margin={"b": 16}, legend_traceorder="normal")
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False)
show(fig, "6-sessions-ended")
display(ended.rename("sessions").to_frame().assign(share=lambda d: (100 * d["sessions"] / total).round(1)))

## 7. Campaigns: sessions that sent the same commands

A campaign here is every shell session that sent exactly the same command lines in the same
order. Before comparing, the random tag each bot puts at the end of its writable-folder probe (six
letters sent as hex escapes to `echo -e`) becomes `<tag>`, so the 25 probe variants count as one.
Fixed markers stay as they are, because they identify the bot: BusyBox names such as `HISILICON`
and `SORA`, and hex strings that repeat across IPs. Sessions that sent nothing aren't a campaign
and are counted separately.

Exact matching splits a campaign when the 30-second limit cuts a session short. When every session
in a campaign hit the limit and it's the start of a longer campaign from the same IP, the
**start of** column names that longer one. The sequence column shows short commands defanged,
names long lines such as downloads by their playbook step, and decodes hex echoes into text.

In [ ]:
# Each bot's writable-folder probe ends with a random six-letter tag sent as hex escapes. The log
# escapes backslashes and quotes, so it's stored as \'\\x41\\x42...\' with six escapes.
RANDOM_TAG = re.compile(r"(?<=echo -e \\')(?:\\\\x[0-9a-fA-F]{2}){6}(?=\\')")
HEX_ESCAPE = re.compile(r"\\\\x([0-9a-fA-F]{2})")
SHORT_STEP = {"menu_escape": "menu escape", "router_cli": "router CLI", "ping_injection": "ping injection",
              "look_around": "look around", "writable_probe": "writable-folder probe",
              "busybox_tag": "BusyBox check", "echo_marker": "echo check", "shell_test": "SHELL_TEST",
              "download": "download and run"}


def normalize(cmd):
    return RANDOM_TAG.sub("<tag>", cmd)


def show_command(cmd):
    """One command, safe to show. Long lines and downloads are named by their step instead."""
    if cmd == "":
        return "(blank line)"
    fam = family(cmd)
    decoded = HEX_ESCAPE.sub(lambda m: chr(int(m[1], 16)), cmd)
    if fam in ("writable_probe", "download") or len(decoded) > 44:
        return f"⟨{SHORT_STEP.get(fam, 'long line')}⟩"
    return defang(decoded, 44) or "(not text)"


def step_summary(seq):
    """A campaign's steps in the order they first appear, with repeats counted."""
    counts = {}
    for cmd in seq:
        name = SHORT_STEP.get(family(cmd)) or ("blank line" if cmd == "" else "garbled")
        counts[name] = counts.get(name, 0) + 1
    text = ", ".join(name if n == 1 else f"{name} ×{n}" for name, n in counts.items())
    return text + (" · NUL-ended" if any(c.endswith("\\0") for c in seq) else "")


sessions["sequence"] = sessions["commands"].map(lambda cs: tuple(normalize(c) for c in cs))
sent_something = sessions[sessions["sequence"].map(len) > 0]
campaigns = (
    sent_something.groupby("sequence")
    .agg(sessions=("ip_id", "size"), IPs=("ip_id", "nunique"), first_seen=("time", "min"),
         last_seen=("time", "max"), downloaded=("downloaded", "sum"),
         hit_limit=("ended", lambda e: int((e == "timed_out").sum())), ip_set=("ip_id", lambda s: set(s)))
    .reset_index()
    .sort_values(["sessions", "IPs", "first_seen"], ascending=[False, False, True], ignore_index=True)
)
campaigns.insert(0, "campaign", [f"C{i}" for i in range(1, len(campaigns) + 1)])


def start_of(row):
    """The longer campaign this one starts, when every session hit the limit and an IP is shared."""
    if row["hit_limit"] < row["sessions"]:
        return ""
    n = len(row["sequence"])
    for _, longer in campaigns.iterrows():
        if (len(longer["sequence"]) > n and longer["sequence"][:n] == row["sequence"]
                and row["ip_set"] & longer["ip_set"]):
            return longer["campaign"]
    return ""


campaigns["start of"] = campaigns.apply(start_of, axis=1)
cut = campaigns[campaigns["start of"] != ""]
raw_sequences = sent_something["commands"].map(tuple).nunique()
print(
    f"{len(campaigns)} campaigns in {len(sent_something)} sessions from {sent_something['ip_id'].nunique()} IPs "
    f"({raw_sequences} exact sequences before the probe tag was replaced). "
    f"{len(sessions) - len(sent_something)} more sessions logged in and sent nothing. "
    f"{int((campaigns['IPs'] == 1).sum())} campaigns came from a single IP. "
    f"{len(cut)} campaigns ({int(cut['sessions'].sum())} sessions) are the start of a longer one from the "
    "same IP and always hit the time limit, so they're probably that campaign cut off."
)

top = campaigns.head(10)
labels = [f"{c} · {step_summary(s)}" for c, s in zip(top["campaign"], top["sequence"])]
fig = go.Figure(go.Bar(
    x=top["sessions"], y=labels, orientation="h", marker={"color": T["s1"]},
    text=[f"{n}  ·  {k} IP{'s' if k != 1 else ''}" for n, k in zip(top["sessions"], top["IPs"])],
    textposition="outside", textfont={"color": T["ink"]}, cliponaxis=False,
    hovertemplate="%{y}<br>%{x} sessions<extra></extra>",
))
style(fig, "The ten biggest campaigns",
      f"Shell sessions that sent exactly the same commands, {WINDOW}, with each bot's random probe tag "
      f"removed. {len(campaigns)} campaigns in all; most came from one IP.", height=520)
fig.update_layout(bargap=0.4, margin={"r": 110})
fig.update_yaxes(autorange="reversed", showgrid=False, ticks="", showline=False, tickfont={"color": T["ink2"]})
fig.update_xaxes(title_text="Sessions", showline=True)
show(fig, "7-campaigns")

campaign_table = pd.DataFrame({
    "campaign": campaigns["campaign"], "sessions": campaigns["sessions"], "IPs": campaigns["IPs"],
    "commands": campaigns["sequence"].map(len), "steps": campaigns["sequence"].map(step_summary),
    "first seen": campaigns["first_seen"].dt.strftime("%b %d %H:%M"),
    "last seen": campaigns["last_seen"].dt.strftime("%b %d %H:%M"),
    "downloaded": campaigns["downloaded"].astype(int), "hit the 30 s limit": campaigns["hit_limit"],
    "start of": campaigns["start of"],
    "sequence (defanged)": campaigns["sequence"].map(lambda s: " › ".join(map(show_command, s))),
})
with pd.option_context("display.max_colwidth", 200, "display.max_rows", 100):
    display(campaign_table)

## 8. Devices the logins were made for

A factory-default login shows what kind of device a bot hopes to find. A username or login below is
matched to a device only where a public source says it's that device's default, and every match
names its source. **documented** means the maker's own manual or a CVE advisory, **reported** means
one security-research write-up, and **uncertain** means the source is weak or only indirect. Generic
names such as `root` and `admin` are defaults on many products, so they aren't matched to any one.
Names with no public source say so rather than guess.

The loader drops passwords, so attempted logins are matched by username alone, and the log can't
say which accepted pair each shell login used. Usernames follow the rule from section 5 (only those
tried by at least two IPs are shown), and any IPv4 address inside one becomes `<IP>`, because the
MGLNDD scanner puts the honeypot's own address there.

In [ ]:
SOURCES = {
    "Krebs 2016": ("Brian Krebs, “Who Makes the IoT Things Under Attack?”, KrebsOnSecurity, Oct 2016, and its "
                   "table of Mirai's less generic logins. He warns the matches “should not be taken as gospel”",
                   "https://krebsonsecurity.com/2016/10/who-makes-the-iot-things-under-attack/"),
    "Huawei manual": ("Huawei EchoLife HG8010/HG8110/HG8240/HG8245/HG8247/HG8447 GPON Terminal Service Manual, "
                      "issue 02 (2011), printed page 145",
                      "https://www.manualslib.com/manual/3321047/Huawei-Echolife-Hg8110.html?page=153"),
    "C-Data manual": ("C-Data FD612H GPON ONT user manual, page 6",
                      "https://www.manualslib.com/manual/1245804/C-Data-Fd612h.html?page=6"),
    "Trend Micro 2018": ("Trend Micro, “Mirai-like Scanning Activity Detected from China, Targets Brazil”, Apr 2018",
                         "https://www.trendmicro.com/en_us/research/18/d/mirai-like-scanning-activity-detected-from-china-targets-in-brazil.html"),
    "Securin 2026": ("Securin, CVE-2026-42372: hardcoded telnet backdoor in D-Link DIR-605L A1, Apr 2026",
                     "https://www.securin.io/zero-day/cve-2026-42372-hardcoded-telnet-backdoor-in-d-link-dir-605l-a1-end-of-life-"),
    "Juniper manual": ("Juniper Networks NetScreen-5GT user's guide (Spanish edition), printed page 14",
                       "https://manualslib.mx/manual/484504/Juniper-Networks-Netscreen-5Gt.html?page=18"),
    "Cisco": ("Cisco, “Configure Password Settings on a Switch through the CLI”, for the Sx300 to Sx550X series",
              "https://www.cisco.com/c/en/us/support/docs/smb/switches/cisco-small-business-300-series-managed-switches/smb5563-configure-password-settings-on-a-switch-through-the-command.html"),
    "Raspberry Pi 2022": ("Simon Long, “An update to Raspberry Pi OS Bullseye”, Raspberry Pi, Apr 2022",
                          "https://www.raspberrypi.com/news/raspberry-pi-bullseye-update-april-2022/"),
    "Antiy 2025": ("Antiy CERT, “Analysis of Three Variants of the HailBot Botnet Attacking DeepSeek”, Feb 2025. "
                   "Its device column is labelled as DeepSeek output with manual revision",
                   "https://www.antiy.net/p/analysis-of-three-variants-of-the-hailbot-botnet-attacking-deepseek/"),
    "Nmap probes": ("Nmap's nmap-service-probes file: the GetRequest, Help, SIPOptions, RPCCheck and NCP probes",
                    "https://github.com/nmap/nmap/blob/master/nmap-service-probes"),
    "SANS ISC 2022": ("SANS Internet Storm Center diary 28458, “MGLNDD_* Scans”, Mar 2022. It describes the "
                      "pattern; who runs the scanner isn't known", "https://isc.sans.edu/diary/28458"),
    "RFC 8446": ("RFC 8446 (TLS 1.3), section 5.1: a record starting with byte 22 is a handshake",
                 "https://www.rfc-editor.org/rfc/rfc8446#section-5.1"),
    "RFC 9112": ("RFC 9112 (HTTP/1.1), section 3: the request line", "https://www.rfc-editor.org/rfc/rfc9112#section-3"),
}

FIBRE, ROUTERS, CAMERAS, BUSINESS, BOARDS = ("ISP fibre modems and gateways", "Routers and wireless gear",
                                             "Cameras and video recorders", "Business network gear",
                                             "Single-board computers")
GENERIC_GROUP, UNMAPPED, NOT_LOGIN = "Generic or system account", "No public mapping found", "Not a login"
# username: (group, device, source, certainty)
USERNAME_DEVICES = {
    "telecomadmin": (FIBRE, "Huawei EchoLife GPON fibre modems (HG8245 and others)", "Huawei manual", "documented"),
    "CUAdmin": (FIBRE, "C-Data FD612H GPON fibre modem", "C-Data manual", "documented"),
    "cuadmin": (FIBRE, "Probably CUAdmin: the manual spells it with capitals", "C-Data manual", "uncertain"),
    "telnetadmin": (FIBRE, "China Telecom home gateway, model E-140W-P", "Trend Micro 2018", "reported"),
    "e8telnet": (FIBRE, "China Telecom home gateway, model HGU421v3", "Trend Micro 2018", "reported"),
    "e8ehome": (FIBRE, "China Telecom home gateway, model E8C", "Trend Micro 2018", "reported"),
    "Alphanetworks": (ROUTERS, "D-Link routers made by Alpha Networks: a telnet backdoor (DIR-605L A1 and others)",
                      "Securin 2026", "documented"),
    "ubnt": (ROUTERS, "Ubiquiti AirOS routers", "Krebs 2016", "reported"),
    "supervisor": (CAMERAS, "VideoIQ cameras", "Krebs 2016", "reported"),
    "666666": (CAMERAS, "Dahua IP cameras", "Krebs 2016", "reported"),
    "888888": (CAMERAS, "Dahua: the source has 888888 as a DVR password for root, not as a username",
               "Krebs 2016", "uncertain"),
    "hikvision": (CAMERAS, "Hikvision devices", "Antiy 2025", "uncertain"),
    "vstarcam2015": (CAMERAS, "VStarcam cameras", "Antiy 2025", "uncertain"),
    "netscreen": (BUSINESS, "Juniper NetScreen firewalls", "Juniper manual", "documented"),
    "cisco": (BUSINESS, "Cisco small-business switches (Sx300 to Sx550X)", "Cisco", "documented"),
    "pi": (BOARDS, "Raspberry Pi OS before April 2022", "Raspberry Pi 2022", "documented"),
}
# Defaults on many products, or standard Unix accounts: not matched to any one device.
GENERIC = {"root", "admin", "default", "guest", "support", "user", "administrator", "Administrator", "Admin",
           "super", "superadmin", "Manager", "operator", "tech", "service", "telnet", "test", "test1", "admin1",
           "mother", "1234", "ftp", "bin", "daemon", "adm", "nobody", "postgres", "ubuntu"}
# (user, password) from credentials.txt: (group, device, source, certainty)
LOGIN_DEVICES = {
    ("root", "admin"): (CAMERAS, "IPX-DDK network camera", "Krebs 2016", "reported"),
    ("root", ""): (CAMERAS, "Vivotek IP camera", "Krebs 2016", "reported"),
    ("admin", "123456"): (CAMERAS, "ACTi IP camera", "Krebs 2016", "reported"),
}
# Four numbers joined by dots, dashes or underscores. Not \b: in MGLNDD_192.0.2.1_23 the address
# follows an underscore, which counts as part of a word.
ANY_IPV4 = re.compile(r"(?<!\d)\d{1,3}(?:[._-]\d{1,3}){3}(?!\d)")


def not_a_login(text):
    """(what it is, source, certainty) when a 'username' is another protocol's first line."""
    if text.startswith("GET / HTTP/1.0"):
        return "Nmap's GetRequest probe (HTTP)", "Nmap probes", "exact match"
    if text == "HELP":
        return "Nmap's Help probe", "Nmap probes", "exact match"
    if text.startswith("OPTIONS sip:nm "):
        return "Nmap's SIPOptions probe (SIP)", "Nmap probes", "exact match"
    if text.startswith("DmdT\\0\\0\\0\\u{17}"):
        return "Nmap's NCP probe (NetWare)", "Nmap probes", "exact match"
    if text.startswith("\\0\\0(r\\u{1d}\\u{13}"):
        return "Nmap's RPCCheck probe (Sun RPC)", "Nmap probes", "exact match"
    if text.startswith("\\u{16}\\u{3}"):
        return "TLS handshake (a ClientHello)", "RFC 8446", "exact match"
    if re.fullmatch(r"(GET|HEAD|POST) \S+ HTTP/1\.[01]", text):
        return "HTTP request", "RFC 9112", "exact match"
    if text.startswith("MGLNDD_"):
        return "MGLNDD scanner: its first line names the address it scanned", "SANS ISC 2022", "reported"
    if not re.search(r"[A-Za-z0-9]", text):
        return "Binary data", "", ""
    return None


def classify(username):
    """(group, device, source, certainty) for one logged username."""
    text = printable(username)
    probe = not_a_login(text)
    if probe:
        return NOT_LOGIN, *probe
    name = text[:-2] if text.endswith("\\0") else text  # some bots end the name with a NUL byte
    if name in USERNAME_DEVICES:
        return USERNAME_DEVICES[name]
    if name in GENERIC:
        return GENERIC_GROUP, "Not tied to one device type", "", ""
    return UNMAPPED, "", "", ""


def shown_name(username):
    text = ANY_IPV4.sub("<IP>", printable(username))[:32]
    return text if re.search(r"[A-Za-z0-9]", text) else "(binary)"


# The logins the shell accepts, from the committed file of factory defaults.
accepted_logins = [line.split(":", 1) for line in (REPO / "deploy/host/credentials.txt").read_text().splitlines()
                   if line.strip() and not line.lstrip().startswith("#")]
login_table = pd.DataFrame([
    {"login": f"{u}:{p or '(empty)'}", **dict(zip(["group", "device", "source", "certainty"],
                                                  LOGIN_DEVICES.get((u, p), (GENERIC_GROUP, "Not tied to one device type", "", ""))))}
    for u, p in accepted_logins
])

named = after[after["username"].fillna("") != ""].copy()
named[["group", "device", "source", "certainty"]] = pd.DataFrame(
    named["username"].map(classify).tolist(), index=named.index)
by_name = (named.groupby("username")
           .agg(IPs=("ip_id", "nunique"), attempts=("ip_id", "size"), group=("group", "first"),
                device=("device", "first"), source=("source", "first"), certainty=("certainty", "first"))
           .query("IPs >= 2").sort_values(["IPs", "attempts"], ascending=False))
by_name.index = [shown_name(u) for u in by_name.index]
by_name.index.name = "username"
# A broader check than the mask, so a gap in one isn't hidden by the other.
assert not any(re.search(r"\d{1,3} ?[._ -] ?\d{1,3} ?[._ -] ?\d{1,3} ?[._ -] ?\d{1,3}", u)
               for u in by_name.index), "an address is showing in a username"

DEVICE_GROUPS = [FIBRE, CAMERAS, ROUTERS, BUSINESS, BOARDS]
by_group = named.groupby("group").agg(IPs=("ip_id", "nunique"), attempts=("ip_id", "size"))
by_group = by_group.reindex(
    sorted(DEVICE_GROUPS, key=lambda g: -by_group["IPs"].get(g, 0)) + [GENERIC_GROUP, UNMAPPED, NOT_LOGIN]
).fillna(0).astype(int)
certain = named[named["certainty"] != "uncertain"].groupby("group")["ip_id"].nunique()
by_group["IPs without uncertain matches"] = certain.reindex(by_group.index).fillna(0).astype(int)
by_group["share of IPs that sent a name"] = (100 * by_group["IPs"] / named["ip_id"].nunique()).round(1)

# Solid: IPs with at least one documented or reported match. Pale: IPs whose only match is uncertain.
sure, unsure = by_group["IPs without uncertain matches"], by_group["IPs"] - by_group["IPs without uncertain matches"]
fig = go.Figure()
fig.add_bar(x=sure, y=by_group.index, orientation="h", name="Documented or reported",
            marker={"color": [T["s1"] if g in DEVICE_GROUPS else T["muted"] for g in by_group.index]},
            hovertemplate="%{y}: %{x} IPs with a documented or reported match<extra></extra>")
pale = "rgba({},{},{},0.4)".format(*(int(T["s1"][i:i + 2], 16) for i in (1, 3, 5)))
fig.add_bar(x=unsure, y=by_group.index, orientation="h", name="Only an uncertain match",
            marker={"color": pale},
            hovertemplate="%{y}: %{x} more IPs whose only match is uncertain<extra></extra>")
for group, n in by_group["IPs"].items():
    fig.add_annotation(x=n, y=group, text=str(n), showarrow=False, xanchor="left", xshift=6,
                       font={"color": T["ink"]})
style(fig, "Which devices the logins were made for",
      f"IPs that sent at least one username in each group, {WINDOW}. An IP can be in several groups.<br>"
      "Grey groups name no device: root, admin and the like are defaults on many products.", height=480)
fig.update_layout(barmode="stack", bargap=0.4, margin={"r": 60, "t": 150}, legend_traceorder="normal")
fig.update_yaxes(autorange="reversed", showgrid=False, ticks="", showline=False, tickfont={"color": T["ink2"]})
fig.update_xaxes(title_text="IP addresses", showline=True)
show(fig, "8-devices")
display(by_group)

print("The logins the shell accepts (deploy/host/credentials.txt):")
display(login_table)
logins_by_name = sessions["username"].value_counts()
assert set(logins_by_name.index) <= {u for u, _ in accepted_logins}  # only names from the committed file
print("Shell logins by username: " + ", ".join(f"{u} {n}" for u, n in logins_by_name.items()) + ".")
print("Usernames tried by at least two IPs:")
with pd.option_context("display.max_rows", 200, "display.max_colwidth", 120):
    display(by_name)
used = sorted({s for s in [*by_name["source"], *login_table["source"]] if s})
print("Sources:")
with pd.option_context("display.max_colwidth", 250):
    display(pd.DataFrame([(k, *SOURCES[k]) for k in used], columns=["source", "what it is", "link"]).set_index("source"))

## 9. Where the visiting IPs are registered

Country and network (ASN) for each IP, from MaxMind's free GeoLite2 databases on this computer. No
IP is sent anywhere for this: `load_geo` reads the addresses, looks them up in the local files and
returns only opaque IDs. A country is where an address is registered, not necessarily where the
person behind it is.

To get the databases (it needs a free MaxMind account, so only you can do it):

1. Sign up at <https://www.maxmind.com/en/geolite2/signup> and confirm the email.
2. Signed in, open **Download Databases** and download **GeoLite2 Country** and **GeoLite2 ASN**,
   each in the GZIP format (an `.mmdb` file inside a `.tar.gz`).
3. Put the two `.mmdb` files in `~/GeoIP/`:
   `mkdir -p ~/GeoIP && for f in ~/Downloads/GeoLite2-{Country,ASN}_*.tar.gz; do tar -xzf "$f" -C ~/GeoIP --strip-components=1 --wildcards '*.mmdb'; done`
4. Run this section. It needs `maxminddb` in `.venv`: `.venv/bin/pip install maxminddb==3.2.0`.

MaxMind's licence asks for two things. Replace a database within 30 days of a newer release (the
cell warns when yours is older than that). And credit them wherever the data is used publicly:
*This product includes GeoLite Data created by MaxMind, available from https://www.maxmind.com.*
Never commit the `.mmdb` files.

In [ ]:
GEOIP_DIR = Path.home() / "GeoIP"
GEO_FILES = {"Country": "GeoLite2-Country.mmdb", "ASN": "GeoLite2-ASN.mmdb"}


def load_geo(log_dir, geo_dir):
    """Country and ASN for every IP in the logs, keyed by opaque ID, from local files only."""
    paths = {name: geo_dir / file for name, file in GEO_FILES.items()}
    missing = [p.name for p in paths.values() if not p.exists()]
    if missing:
        print(f"Skipped: {' and '.join(missing)} not found in {geo_dir}. The steps above say how to get them.")
        return None, {}
    import maxminddb

    ips = set()
    for path in sorted(log_dir.glob("visits-*.jsonl")):
        with path.open(encoding="utf-8", errors="replace") as f:
            for line in f:
                try:
                    ips.add(json.loads(line)["source_ip"])
                except (json.JSONDecodeError, KeyError):
                    pass
    rows, built = {}, {}
    with maxminddb.open_database(paths["Country"]) as country_db, maxminddb.open_database(paths["ASN"]) as asn_db:
        for name, db in [("Country", country_db), ("ASN", asn_db)]:
            built[name] = datetime.fromtimestamp(db.metadata().build_epoch, timezone.utc)
        for ip in ips:
            try:
                place, network = country_db.get(ip) or {}, asn_db.get(ip) or {}
            except ValueError:
                continue  # not an IP address
            country = place.get("country") or place.get("registered_country") or {}
            rows[opaque_id(ip)] = {
                "country": country.get("iso_code") or "??",
                "country name": (country.get("names") or {}).get("en", "Unknown"),
                "ASN": network.get("autonomous_system_number"),
                "AS organization": network.get("autonomous_system_organization") or "Unknown",
            }
    geo = pd.DataFrame.from_dict(rows, orient="index")
    geo["ASN"] = geo["ASN"].astype("Int64")
    return geo, built


geo, geo_built = load_geo(LOG_DIR, GEOIP_DIR)
if geo is not None:
    for name, when in geo_built.items():
        age = (datetime.now(timezone.utc) - when).days
        print(f"GeoLite2 {name}: built {when:%b %d, %Y}"
              + (f", {age} days ago. Download the current one: the licence gives 30 days to replace an old copy."
                 if age > 30 else "."))
    placed = after.join(geo, on="ip_id")
    n_ips = after["ip_id"].nunique()

    def per(columns):
        table = placed.groupby(columns, dropna=False).agg(
            IPs=("ip_id", "nunique"), visits=("ip_id", "size"),
            **{"shell sessions": ("outcome", lambda o: int((o == "logged_in").sum()))})
        table["share of IPs %"] = (100 * table["IPs"] / n_ips).round(1)
        return table.sort_values(["IPs", "visits"], ascending=False)

    countries, networks = per(["country", "country name"]), per(["ASN", "AS organization"])
    print(f"{n_ips} IPs from {len(countries)} countries and {len(networks)} networks, {WINDOW}. "
          f"Not in the databases: {int(placed['country'].isna().sum())} visits.")
    top_countries = countries.head(10).reset_index()
    fig = go.Figure(go.Bar(
        x=top_countries["IPs"], y=top_countries["country name"], orientation="h", marker={"color": T["s1"]},
        text=top_countries["IPs"], textposition="outside", textfont={"color": T["ink"]}, cliponaxis=False,
        customdata=top_countries[["visits", "shell sessions"]],
        hovertemplate="%{y}: %{x} IPs, %{customdata[0]:,} visits, %{customdata[1]} shell sessions<extra></extra>",
    ))
    style(fig, "Where the visiting IPs are registered",
          f"Top 10 countries by IPs, {WINDOW}. Registered country, not where the person behind it is. "
          "Data: MaxMind GeoLite2.", height=460)
    fig.update_layout(bargap=0.4)
    fig.update_yaxes(autorange="reversed", showgrid=False, ticks="", showline=False, tickfont={"color": T["ink2"]})
    fig.update_xaxes(title_text="IP addresses", showline=True)
    show(fig, "9-countries")
    display(countries.head(15))
    display(networks.head(15))

## 10. Were they already known? URLhaus and AbuseIPDB

[`threat_intel_lookup.py`](threat_intel_lookup.py) asks two services whether they already knew the
download addresses and the IPs. You run it by hand, and this notebook only reads what it saved. It
sends each download address to URLhaus's URL lookup, each address's server to URLhaus's host
lookup, and each visitor IP to AbuseIPDB's check lookup, with nothing else from the logs. It never
submits or reports anything, never contacts an address from the logs, and refuses redirects.
Without `--send` it only shows what it would look up.

1. Get a free URLhaus Auth-Key at <https://auth.abuse.ch/> and a free AbuseIPDB API key at
   <https://www.abuseipdb.com/account/api>.
2. From the repository root, `python3 analysis/threat_intel_lookup.py` shows what it would send.
   `python3 analysis/threat_intel_lookup.py --send` asks for both keys without showing them and
   saves to `~/honeypot-logs/intel/lookups.json`. A free AbuseIPDB account has 1,000 checks a day,
   and a second run carries on where the first stopped.
3. `python3 analysis/abuse_review.py --scanners --since 2026-11-02` marks the research scanners
   among the visitors, the same way the reporter skips them. It makes reverse DNS lookups and
   nothing else, and saves to `~/honeypot-logs/intel/scanners.json`. That file holds full IPs
   too, and nothing deletes it, so delete it with the lookup results when you're done.
4. Run this section again.

The last part works out every figure in the report's "Threat intel" section, over all the IPs the
script checked, and the check below compares them with the report.

"Before us" compares the date URLhaus added a URL or first saw its server, and the times of
AbuseIPDB's reports, with the first time the honeypot saw that address or IP. AbuseIPDB returns
only the past 365 days of reports, so "first reported" means first within that year. It also lists
only the newest 100 reports per IP. For an IP with more, the oldest are missing, so "before us" and
the days before us are lower bounds, and when every listed report came after us it can't be told
whether an older one came before. For those, `threat_intel_lookup.py --older-reports` makes one
more check that asks only how many reports came in since a day or more before the honeypot first
saw the IP: if the year check counted more, the rest are older. Its docs say the list holds up to
10,000, but no lookup here listed more than 100. The results file holds full IPs, so it stays with the logs. The script drops entries older than 30 days each
time it runs, but `deploy/daily-logs.sh` doesn't delete it, so delete it yourself when you're done.

In [ ]:
INTEL_FILE = LOG_DIR / "intel" / "lookups.json"
YES_NO = {"ok": "yes", "no_results": "no"}
# The campaigns the report names, picked out by what they sent (section 7 lists them all).
NUL_BOT = [("enable\\0",), ("enable\\0", "linuxshell\\0"), ("enable\\0", "linuxshell\\0", "system\\0"),
           ("enable\\0", "linuxshell\\0", "system\\0", "shell\\0")]
# The second NUL-word bot, from Nov 10: no linuxshell, and up to sh.
NUL_BOT_2 = [("enable\\0", "system\\0"), ("enable\\0", "system\\0", "shell\\0"),
             ("enable\\0", "system\\0", "shell\\0", "sh\\0")]
REPORTED_FILE = LOG_DIR / "abuseipdb" / "reported.json"
SCANNERS_FILE = LOG_DIR / "intel" / "scanners.json"
REPORT_CAMPAIGNS = {
    "router-CLI and writable-folder probe": lambda s: any(is_marker_probe(c) for c in s),
    "enable … ping ;sh opening": lambda s: s == ("enable", "system", "shell", "sh", "linuxshell", "ping ;sh",
                                                "/bin/busybox"),
    "HISILICON BusyBox check": lambda s: "/bin/busybox HISILICON" in s,
    "uname -m then cat /proc": lambda s: s == ("uname -m", "cat /proc"),
    "echo SHELL_TEST": lambda s: s == ("echo SHELL_TEST",),
    "NUL-ended menu words": lambda s: s in NUL_BOT,
    "NUL-ended menu words, second bot": lambda s: s in NUL_BOT_2,
}
PERSISTENT = ["uname -m then cat /proc", "echo SHELL_TEST", "NUL-ended menu words", "NUL-ended menu words, second bot"]
# AbuseIPDB's check lists only the newest 100 reports per IP. In the Nov 10 lookups no list held more,
# and every IP with more than 100 reports listed exactly 100, including our own recent reports.
LISTED_REPORTS = 100


def load_intel(path):
    """The lookup script's results. Every IP and address becomes an opaque ID as it's read."""
    if not path.exists():
        return None, None
    data = json.loads(path.read_text())
    addresses = {}
    for address, r in data.get("urlhaus", {}).items():
        url, host = r.get("url") or {}, r.get("host") or {}
        addresses[opaque_id(address)] = {
            "address (defanged)": defang(address, 70),
            "first seen here": pd.Timestamp(r["first_seen_here"]),
            "URLhaus had the URL": "(a bare server)" if r["kind"] == "host" else YES_NO.get(url.get("status"), url.get("status")),
            "URL added": url.get("date_added"), "URL before us": url.get("before_us"),
            "URL status": url.get("url_status"),
            "malware": printable(", ".join(url.get("signatures") or []))[:40] or None,
            "tags": printable(", ".join(url.get("tags") or []))[:40] or None,
            "URLhaus had the server": YES_NO.get(host.get("status"), host.get("status")),
            "server first seen": host.get("firstseen"), "server before us": host.get("before_us"),
            "URLs on the server": host.get("url_count"),
        }
    ips, older = {}, data.get("abuseipdb_older", {})
    for ip, r in data.get("abuseipdb", {}).items():
        # The second check (--older-reports) counts only if it came after this year check.
        counted = older.get(ip, {}).get("year_checked_at") == r["checked_at"]
        ips[opaque_id(ip)] = {
            "first seen here": pd.Timestamp(r["first_seen_here"]),
            "checked": pd.Timestamp(r["checked_at"]),
            "score": r.get("score"), "reports": r.get("total_reports") or 0,
            "reporters": r.get("distinct_users"),
            "first reported": pd.Timestamp(r["first_reported"]) if r.get("first_reported") else pd.NaT,
            "reports before us": r.get("reports_before_us") or 0,
            "usage type": r.get("usage_type"), "Tor": bool(r.get("is_tor")),
            "counted": counted, "older by count": counted and bool(older[ip]["older"]),
            "second checked": pd.Timestamp(older[ip]["checked_at"]) if counted else pd.NaT,
            # Reports that came in after the year check count only in the second check.
            "second count higher": counted and older[ip]["recent_total"] > older[ip]["year_total"],
        }
    return pd.DataFrame.from_dict(addresses, orient="index"), pd.DataFrame.from_dict(ips, orient="index")


def load_first_reporter(path):
    """The --first-reporter results, keyed by opaque ID: who reported each IP first, and when ours went."""
    columns = ["first", "our report at", "ours listed", "listed", "reports", "counted", "older by count",
               "second count higher"]
    if not path.exists():
        return pd.DataFrame(columns=columns)
    data = json.loads(path.read_text())
    found, older = data.get("first_reporter", {}), data.get("first_reporter_older", {})

    def counted(ip, r):  # the second check, made after this lookup
        return older.get(ip, {}).get("year_checked_at") == r["checked_at"]

    return pd.DataFrame.from_dict({opaque_id(ip): {"first": r["first"], "our report at": r["our_report_at"],
                                                   "ours listed": r["ours_found"], "listed": r["listed"],
                                                   "reports": r["total_reports"], "counted": counted(ip, r),
                                                   "older by count": counted(ip, r) and bool(older[ip]["older"]),
                                                   "second count higher": counted(ip, r) and (
                                                       older[ip]["recent_total"] > older[ip]["year_total"])}
                                   for ip, r in found.items()}, orient="index", columns=columns)


def load_reported(path):
    """The reporter's record. Every IP becomes an opaque ID as it's read."""
    if not path.exists():
        return None
    reports = json.loads(path.read_text()).get("reports", {})
    return pd.DataFrame([{"ip_id": opaque_id(ip), "reported_at": r["reported_at"]} for ip, r in reports.items()],
                        columns=["ip_id", "reported_at"]).set_index("ip_id")


def load_scanners(path):
    """abuse_review.py --scanners: each IP's research scanner, if any. Every IP becomes an opaque ID as it's read."""
    columns = ["scanner", "checked"]
    if not path.exists():
        return pd.DataFrame(columns=columns)
    found = json.loads(path.read_text()).get("ips", {})
    return pd.DataFrame.from_dict({opaque_id(ip): {"scanner": r["scanner"], "checked": r["checked"]}
                                   for ip, r in found.items()}, orient="index", columns=columns)


def day(moment):
    return f"{moment:%b} {moment.day}"


intel_numbers = None  # the report's threat-intel figures, compared by the check below
# Every IP that visited from the fake login going live on Nov 2 to the end of the report's window.
window_ips = set(visits[(visits["time"] >= STAGE6_OPEN) & in_range]["ip_id"])
intel_addresses, intel_ips = load_intel(INTEL_FILE)
if intel_ips is not None and not intel_ips.empty:
    intel_ips = intel_ips[intel_ips.index.isin(window_ips)]  # results for later IPs wait for a later report
if intel_addresses is None:
    print(f"No lookup results yet ({INTEL_FILE} doesn't exist). Run python3 analysis/threat_intel_lookup.py "
          "from the repository root as described above, then this cell again.")
else:
    # The file can hold one service's results without the other's (--only).
    if intel_addresses.empty:
        print("No URLhaus results in the file yet: run the script with --send and no --only once "
              "you have a URLhaus Auth-Key.")
    else:
        # Sessions and IPs behind each address. A session that named an address twice counts once.
        behind = sessions[["ip_id", "url_ids"]].explode("url_ids").dropna(subset=["url_ids"]).reset_index()
        behind = behind.groupby("url_ids").agg(sessions=("index", "nunique"), IPs=("ip_id", "nunique"))
        address_table = intel_addresses.join(behind).sort_values("first seen here")
        address_table["first seen here"] = address_table["first seen here"].dt.strftime("%b %d %H:%M")
        known = address_table[(address_table["URLhaus had the URL"] == "yes") | (address_table["URLhaus had the server"] == "yes")]
        print(f"URLhaus knew {len(known)} of {len(address_table)} download addresses (the URL or its server).")
        with pd.option_context("display.max_colwidth", 80):
            display(address_table.reset_index(drop=True))

    if intel_ips.empty:
        print("No AbuseIPDB results in the file yet.")
    else:
        groups = {f"All IPs, {WINDOW}": set(after["ip_id"]), "Got into the shell": set(sessions["ip_id"]),
                  "Reached a download": set(sessions[sessions["downloaded"]]["ip_id"])}
        # For an IP with more reports than AbuseIPDB lists, the oldest are missing. When every listed
        # report came after the honeypot saw it, it can't be told whether an older one came before.
        listed_early = intel_ips["reports before us"] > 0  # a listed report from before
        # Or the second check counted reports from before, where the list couldn't show them.
        seen_early = listed_early | intel_ips["older by count"]
        cut_off = intel_ips["reports"] > LISTED_REPORTS
        cant_tell = cut_off & ~seen_early
        not_before = ~cut_off & ~seen_early  # the whole year is listed, and nothing came before us
        rows = []
        for label, ids in groups.items():
            found = intel_ips[intel_ips.index.isin(ids)]
            early = found[found["reports before us"] > 0]
            lead = (early["first seen here"] - early["first reported"]).dt.days
            rows.append({
                "group": label, "IPs": len(ids), "checked": len(found),
                "reported in the past year": int((found["reports"] > 0).sum()),
                "reported before we saw them": int(seen_early.loc[found.index].sum()),
                "can't be told (list cut off)": int(cant_tell.loc[found.index].sum()),
                "median days reported before us": lead.median() if len(lead) else None,
                "median score": found["score"].median() if len(found) else None,
                "Tor exits": int(found["Tor"].sum()),
            })
        display(pd.DataFrame(rows).set_index("group"))

        downloaders = sessions[sessions["downloaded"]].groupby("ip_id").agg(
            IP=("ip_masked", "first"), sessions=("ip_masked", "size"))
        downloaders = downloaders.join(intel_ips[["score", "reports", "first reported", "reports before us",
                                                  "older by count", "checked", "usage type"]])
        downloaders["days from first report to the check"] = (
            (downloaders["checked"] - downloaders["first reported"]).dt.total_seconds() / 86400).round(1)
        downloaders["first reported"] = downloaders["first reported"].dt.strftime("%b %d, %Y")
        print("The IPs that reached a download (masked):")
        display(downloaders.drop(columns="checked").reset_index(drop=True))
        print(f"Usage types AbuseIPDB gives for the IPs, {WINDOW}:")
        display(intel_ips[intel_ips.index.isin(groups[f"All IPs, {WINDOW}"])]["usage type"].fillna("not given").value_counts().to_frame("IPs"))

        # The figures in the report's "Threat intel" section, over every IP the script checked.
        n_checked = len(intel_ips)
        ever_reported = intel_ips["reports"] > 0
        # Only where a report from before is listed: the count doesn't say how long before.
        lead_days = (intel_ips.loc[listed_early, "first seen here"] - intel_ips.loc[listed_early, "first reported"]).dt.days
        high_score = intel_ips["score"] >= 75
        in_shell = pd.Series(intel_ips.index.isin(set(sessions["ip_id"])), index=intel_ips.index)
        n_shell = int(in_shell.sum())
        # Research scanners, matched as the reporter matches them. The list isn't complete, so "at least".
        scanner_list = load_scanners(SCANNERS_FILE)
        is_scanner = pd.Series(intel_ips.index.isin(scanner_list.index[scanner_list["scanner"].notna()]),
                               index=intel_ips.index)

        def shares(ids):
            """Reported in the year, before the honeypot saw them, and at most before, in percent, unrounded."""
            return [100 * int((ids & m).sum()) / int(ids.sum()) for m in (ever_reported, seen_early, seen_early | cant_tell)]

        print(f"\nAll {n_checked} IPs checked, first seen {day(intel_ips['first seen here'].min())} to "
              f"{day(intel_ips['first seen here'].max())}: {int(ever_reported.sum())} reported in the past year, "
              f"{int(seen_early.sum())} before the honeypot first saw them (median {lead_days.median():.0f} days "
              f"earlier) and {int(cant_tell.sum())} more that can't be told, {int(high_score.sum())} scoring 75 or more.")
        spread = pd.cut(intel_ips["score"], [-1, 0, 24, 49, 74, 99, 100],
                        labels=["0", "1–24", "25–49", "50–74", "75–99", "100"]).value_counts().sort_index()
        display(spread.rename("IPs").to_frame().rename_axis("abuse confidence score"))

        rows, campaign_ips = [], {}
        for name, test in REPORT_CAMPAIGNS.items():
            found = campaigns[campaigns["sequence"].map(test)]
            members = set().union(*found["ip_set"])
            campaign_ips[name] = members
            looked_up = intel_ips[intel_ips.index.isin(members)]
            rows.append({"campaign": name, "IDs": ", ".join(found["campaign"]),
                         "sessions": int(found["sessions"].sum()), "IPs": len(members),
                         "reported before us": int(seen_early.loc[looked_up.index].sum()),
                         "can't be told": int(cant_tell.loc[looked_up.index].sum()),
                         "lowest score now": looked_up["score"].min()})
        campaign_intel = pd.DataFrame(rows).set_index("campaign")
        print("The campaigns the report names (IDs from section 7):")
        display(campaign_intel)

        download_ids = set(sessions[sessions["downloaded"]]["ip_id"])
        in_downloads = intel_ips.index.isin(download_ids)
        # Each of these has 100 listed reports, all from after the honeypot saw it.
        download_unknown = intel_ips[in_downloads & cant_tell]
        persistent_ips = set().union(*(campaign_ips[name] for name in PERSISTENT))
        persistent = intel_ips[intel_ips.index.isin(persistent_ips)]

        def known_of(name):
            return f"{campaign_intel.loc[name, 'reported before us']} of {campaign_intel.loc[name, 'IPs']}"

        intel_numbers = {
            "IPs checked": n_checked,
            "Every IP in the report's window was checked": window_ips <= set(intel_ips.index),
            "First sightings from": day(intel_ips["first seen here"].min()),
            "First sightings to": day(intel_ips["first seen here"].max()),
            "Checked on": " and ".join(day(t) for t in sorted({t.normalize() for t in intel_ips["checked"]})),
            "Last checked on": day(intel_ips["checked"].max()),
            "Shell IPs": n_shell,
            "Shell IPs reported in the past year": int((in_shell & ever_reported).sum()),
            "Shell IPs reported in the past year %": pct(int((in_shell & ever_reported).sum()), n_shell),
            "Shell IPs reported before the honeypot saw them": int((in_shell & seen_early).sum()),
            "Shell IPs reported before the honeypot saw them %": pct(int((in_shell & seen_early).sum()), n_shell),
            "Shell IPs reported before the honeypot saw them, at most": int((in_shell & (seen_early | cant_tell)).sum()),
            "Shell IPs reported before the honeypot saw them, at most %": pct(
                int((in_shell & (seen_early | cant_tell)).sum()), n_shell),
            "Reported in the past year": int(ever_reported.sum()),
            "Reported in the past year %": pct(int(ever_reported.sum()), n_checked),
            "No reports in the past year": int((~ever_reported).sum()),
            "Every IP in the window had the scanner check": window_ips <= set(
                scanner_list.index[scanner_list["checked"].astype(bool)]),
            "Research scanners, at least": int(is_scanner.sum()),
            "Research scanners that got into the shell": int((is_scanner & in_shell).sum()),
            "Every research scanner was reported before the honeypot saw it": bool(seen_early[is_scanner].all()),
            "Leaving the scanners out moves each share by under a point": all(
                abs(a - b) < 1 for a, b in zip(shares(~is_scanner), shares(is_scanner | ~is_scanner))),
            "Reported before the honeypot saw them": int(seen_early.sum()),
            "Reported before the honeypot saw them %": pct(int(seen_early.sum()), n_checked),
            "Reported before the honeypot saw them, at most": int((seen_early | cant_tell).sum()),
            "Reported before the honeypot saw them, at most %": pct(int((seen_early | cant_tell).sum()), n_checked),
            "Median days reported before the honeypot saw them": int(lead_days.median()),
            "IPs with more reports than the 100 AbuseIPDB lists": int(cut_off.sum()),
            "…of them with no listed report before the honeypot saw them": int((cut_off & ~listed_early).sum()),
            "Every IP the lists couldn't settle had the second check": bool(
                intel_ips.loc[cut_off & ~listed_early, "counted"].all()),
            "…shown reported before by the second check": int((intel_ips["older by count"] & ~listed_early).sum()),
            "…still can't be told after it": int(cant_tell.sum()),
            "Second checks on": " and ".join(day(t) for t in sorted(
                {t.normalize() for t in intel_ips.loc[intel_ips["counted"], "second checked"]})),
            "For each still undecided, the second check counted more than the year check": bool(
                intel_ips.loc[cant_tell, "second count higher"].all()),
            "Reported before the honeypot saw them, from a listed report": int(listed_early.sum()),
            "IPs that tried a download": len(download_ids),
            "Download IPs reported before they arrived": int((in_downloads & seen_early).sum()),
            "Download IPs that can't be told": len(download_unknown),
            "Download IPs with no reports before they arrived": int((in_downloads & not_before).sum()),
            # Every listed report came after the honeypot saw them, so only the second check could show it.
            "Download IPs shown only by the second check": int(
                (in_downloads & intel_ips["older by count"] & ~listed_early).sum()),
            "Probe campaign IPs reported before us": known_of("router-CLI and writable-folder probe"),
            "ping opening IPs reported before us": known_of("enable … ping ;sh opening"),
            "HISILICON IPs reported before us": known_of("HISILICON BusyBox check"),
            "uname -m campaign IPs": int(campaign_intel.loc["uname -m then cat /proc", "IPs"]),
            "echo SHELL_TEST campaign IPs": int(campaign_intel.loc["echo SHELL_TEST", "IPs"]),
            "NUL-word bot IPs": int(campaign_intel.loc["NUL-ended menu words", "IPs"]),
            "Second NUL-word bot IPs": int(campaign_intel.loc["NUL-ended menu words, second bot", "IPs"]),
            "Persistent bots came back dozens of times (24+ sessions each)": bool(
                (campaign_intel.loc[PERSISTENT, "sessions"] >= 24).all()),
            "Persistent bot IPs": len(persistent_ips),
            "Persistent bot IPs reported before us": int(seen_early.loc[persistent.index].sum()),
            "Persistent bot IPs that can't be told": int(cant_tell.loc[persistent.index].sum()),
            "Persistent bot IPs shown only by the second check": int(
                (persistent["older by count"] & ~listed_early.loc[persistent.index]).sum()),
            "Persistent bot IPs scoring 100 now": int((persistent["score"] == 100).sum()),
            "Lowest persistent bot score now": int(persistent["score"].min()),
            "Fixed-line ISP IPs": int((intel_ips["usage type"] == "Fixed Line ISP").sum()),
            "Scoring 75 or more": int(high_score.sum()),
            "Scoring 75 or more %": pct(int(high_score.sum()), n_checked),
            # With the whole year listed, no report before us means none in the year before they arrived.
            "Scoring 75 or more now, with no reports before us": int((high_score & not_before).sum()),
            "Scoring 75 or more now, with no reports before us, at most": int((high_score & (not_before | cant_tell)).sum()),
        }

# The reporter's record, and who reported each of those IPs first (threat_intel_lookup.py --first-reporter).
reported, first_numbers = load_reported(REPORTED_FILE), None
if reported is None:
    print(f"No reports recorded yet ({REPORTED_FILE} doesn't exist).")
else:
    report_day = pd.to_datetime(reported["reported_at"]).dt.strftime("%b %d")
    print(f"Reported to AbuseIPDB: {len(reported)} IPs, " + ", ".join(f"{n} on {d}" for d, n in report_day.value_counts().sort_index().items()) + ".")
    firsts = load_first_reporter(INTEL_FILE)
    current = firsts[firsts.index.isin(reported.index)]
    current = current[current["our report at"] == reported.loc[current.index, "reported_at"]]
    if len(current) < len(reported):
        print(f"First reporters: {len(current)} of the {len(reported)} reported IPs looked up so far. Run "
              "python3 analysis/threat_intel_lookup.py --first-reporter --send for the rest.")
    else:
        counts = current["first"].value_counts()
        unsure = current["first"] == "can't tell"
        by_count = unsure & current["older by count"].astype(bool)
        first_numbers = {"IPs looked up": len(current), "Us first": int(counts.get("us", 0)),
                         "Someone else first": int(counts.get("someone else", 0)),
                         "Can't tell from the lists": int(unsure.sum()),
                         "Every one of those had the second check": bool(current.loc[unsure, "counted"].all()),
                         "…someone else first, shown by the second check": int(by_count.sum()),
                         "…still can't tell after it": int((unsure & ~by_count).sum()),
                         "Someone else first, with the second check": int(counts.get("someone else", 0)) + int(by_count.sum()),
                         "For each still undecided, the second check counted more than the first lookup": bool(
                             current.loc[unsure & ~by_count, "second count higher"].astype(bool).all()),
                         # Ours wasn't among the newest 100 AbuseIPDB lists, out of more than 100.
                         "Can't tell because ours wasn't among the 100 listed": int(
                             ((current["first"] == "can't tell") & ~current["ours listed"].astype(bool)
                              & (current["listed"] == LISTED_REPORTS) & (current["reports"] > LISTED_REPORTS)).sum()),
                         "Us first, all reported on Nov 10": bool(
                             (report_day.loc[current.index[current["first"] == "us"]] == "Nov 10").all())}
        print("Who reported them first, in the past year:", first_numbers)

## Check against README_REPORT.md

Every number the report takes from the logs, recomputed here. The telnet-opening comparison has its
own window, Nov 3 to Nov 6 (`after_telnet`). The threat-intel numbers come from the lookup results
that section 10 reads, so they're checked once every IP in the window has been looked up, and
skipped with a note until then. The same goes for who reported each IP first. The few numbers that
come from elsewhere (server health, data sent, test counts) aren't in the logs, so they aren't
checked.

In [ ]:
creds = [line for line in (REPO / "deploy/host/credentials.txt").read_text().splitlines()
         if line.strip() and not line.lstrip().startswith("#")]
days = list(after.groupby(after["time"].dt.date))
telnet_days = list(after_telnet.groupby(after_telnet["time"].dt.date))
day_silent = [pct(len(silent(g)), len(g), 1) for _, g in days]
telnet_day_silent = [pct(len(silent(g)), len(g), 1) for _, g in telnet_days]
download_lines = cmds[cmds["family"] == "download"]["cmd"]
shell_test_only = sessions[sessions["commands"].map(lambda c: c == ["echo SHELL_TEST"])]
step = playbook.set_index("key")
checked = sessions[sessions["ran_check"]]
mirai_opening = ["enable", "system", "shell", "sh", "linuxshell"]
menu_opening = ["enable", "system", "shell", "sh"]
downloaders = sessions[sessions["downloaded"]]
early = sessions[sessions["time"] <= TELNET_END]  # the shell sessions of the first study, Nov 3 to Nov 6


def days_of(times):
    return ", ".join(day(t) for t in sorted({t.normalize() for t in times}))


def uses(df, test):
    return df["commands"].map(lambda cs: any(test(c) for c in cs))


# Every download address once, with its masked text and its server's opaque ID.
addresses = (sessions[["url_ids", "urls_masked", "url_hosts"]].explode(["url_ids", "urls_masked", "url_hosts"])
             .dropna(subset=["url_ids"]).drop_duplicates("url_ids"))
urls = addresses[addresses["urls_masked"].str.contains("://", regex=False)]
CPU_NAMED = re.compile(r"[_./-](arm[4-7]?|mips|mpsl|mipsel|m68k|ppc|sh4|spc|x86|i[3-6]86)$")
PIPED = re.compile(r"(?<!\|)\|\s*sh\b")


def piped(masked_url):
    """The address is fetched in a command whose output goes straight into sh."""
    return any(masked_url in part and PIPED.search(part) for cs in downloaders["commands"] for c in cs
               for part in mask_address(c).split(";"))


# Each URL as published: defanged, and a payload's own file name under /payload/<id>/ replaced.
def published(masked_url):
    return re.sub(r"(/payload/<id>/)[A-Za-z0-9]+\.sh", r"\1<name>.sh", show_url(masked_url))


def show_url(masked_url):
    return re.sub(r"(\d{1,3})\.(\d{1,3})\.x\.x", r"\1[.]\2[.]x[.]x", defang(masked_url))


seen_at = (sessions[["time", "urls_masked"]].explode("urls_masked").dropna(subset=["urls_masked"])
           .groupby("urls_masked")["time"].min())
url_first_seen = {published(u): t for u, t in seen_at.items() if "://" in u}
url_first_days = {u: day(t) for u, t in url_first_seen.items()}
url_kinds = urls["urls_masked"].map(lambda u: bool(CPU_NAMED.search(urlsplit(u).path) or "/bins/" in u or piped(u)))

RANDOM_WORD = re.compile(r"/bin/busybox (?=[A-Za-z0-9]*[a-z])(?=[A-Za-z0-9]*[A-Z])(?=[A-Za-z0-9]*\d)[A-Za-z0-9]{8}\s*")
FIRST_DETECTION_1 = re.compile(r"busybox [A-Z][A-Za-z0-9]{2,}")  # the report's first version of detection 1
random_word = sessions[uses(sessions, RANDOM_WORD.fullmatch)]
ftp = sessions[uses(sessions, lambda c: "ftpget" in c)]
nul_1 = sessions[sessions["commands"].map(tuple).isin(NUL_BOT)]
nul_2 = sessions[sessions["commands"].map(tuple).isin(NUL_BOT_2)]
rivals_hex = sessions.loc[sorted(rival_scripts)]
killed = KILL_PATTERN.findall("".join(rivals_hex["commands"].map(hex_script)))
CPU_BUILDS = {"arm", "mips", "mpsl", "mipsel", "aarch64", "x86"}
# "||" between two download tools, not just anywhere (a dropper's cd chain uses it too).
CHAINED = re.compile(r"(?:wget|curl|tftp|ftpget)[^;|&]*\|\|[^;]*?(?:wget|curl|tftp|ftpget)")
dropper_tools = downloaders["commands"].map(lambda cs: {t for c in cs for t in ("wget", "curl", "tftp", "ftpget") if t in c})

checks = [
    # The window.
    ("Hours live", 186, round(hours_live)),
    ("Last visit", "Nov 10 19:45", f"{day(after['time'].max())} {after['time'].max():%H:%M}"),
    # Headline numbers, Nov 3 to Nov 10.
    ("Connections", 5527, len(after)),
    ("IP addresses", 790, after["ip_id"].nunique()),
    ("Logins into the fake shell", 839, len(sessions)),
    ("IPs that got in", 330, sessions["ip_id"].nunique()),
    ("Commands", 3784, len(cmds)),
    ("Different commands", 152, cmds["cmd"].nunique()),
    ("Sessions that reached a download", 23, int(sessions["downloaded"].sum())),
    ("IPs that tried a download", 12, downloaders["ip_id"].nunique()),
    ("Download addresses", 154, int(sessions["url_ids"].map(len).sum())),
    ("Unique download addresses", 17, len(addresses)),
    ("Download servers", 9, addresses["url_hosts"].nunique()),
    ("Download URLs", 11, len(urls)),
    ("Servers named on their own, for TFTP or FTP", 6, len(addresses) - len(urls)),
    ("Every download server is a bare IPv4 address", True,
     bool(addresses["urls_masked"].map(lambda u: re.fullmatch(r"\d{1,3}\.\d{1,3}\.x\.x", url_host(u)) is not None).all())),
    ("Download URLs (defanged, with names that could identify the honeypot replaced) and first days",
     "hxxp://160[.]119[.]x[.]x/bins/kla.sh Nov 3; hxxps://217[.]60[.]x[.]x/sh Nov 4; "
     "hxxp://168[.]220[.]x[.]x:9087/payload/<id>/<name>.sh Nov 6; hxxp://201[.]79[.]x[.]x:8081/main_arm7 Nov 6; "
     "hxxp://201[.]79[.]x[.]x:8081/main_mips Nov 6; hxxp://201[.]79[.]x[.]x:8081/main_mpsl Nov 6; "
     "hxxp://176[.]65[.]x[.]x/telnet2.sh Nov 8; hxxp://160[.]187[.]x[.]x/all.sh Nov 9; hxxp://37[.]221[.]x[.]x/all.sh Nov 9; "
     "hxxp://102[.]220[.]x[.]x/curl.sh Nov 10; hxxp://102[.]220[.]x[.]x/wget.sh Nov 10",
     "; ".join(f"{a} {d}" for a, d in sorted(url_first_days.items(), key=lambda kv: (url_first_seen[kv[0]], kv[0])))),
    ("URLs named for a CPU type, under /bins/ or piped into sh", "7 of 11", f"{int(url_kinds.sum())} of {len(urls)}"),
    ("Silent visits", 661, len(silent(after))),
    ("Silent %", 12, pct(len(silent(after)), len(after))),
    ("Factory-default logins accepted", 18, len(creds)),
    ("Sent a login", 4805, len(sent_login(after))),
    ("Sent a login %", 87, pct(len(sent_login(after)), len(after))),
    ("Sessions with a command", 706, int(with_command.sum())),
    ("Download share of sessions %", 2.7, pct(int(sessions["downloaded"].sum()), len(sessions), 1)),
    ("Funnel shares of connections", "86.9%, 15.2%, 12.8%, 6.3%, 0.4%", ", ".join(funnel["share of connections"][1:])),
    ("Sessions that hung up", 531, int((sessions["ended"] == "left").sum())),
    ("Sessions that hung up %", 63, pct(int((sessions["ended"] == "left").sum()), len(sessions))),
    ("Sessions that hit the 30-second limit", 308, int((sessions["ended"] == "timed_out").sum())),
    ("Sessions that hit the 30-second limit %", 37, pct(int((sessions["ended"] == "timed_out").sum()), len(sessions))),
    ("No session typed exit", True, not (sessions["ended"] == "exit").any()),
    ("Sessions that ran a real-shell check", 348, len(checked)),
    ("…and never downloaded", 341, int((~checked["downloaded"]).sum())),
    ("…and downloaded anyway", 7, int(checked["downloaded"].sum())),
    ("Sessions opening with enable, system, shell, sh, linuxshell", 124,
     int(sessions["commands"].map(lambda c: c[:5] == mirai_opening).sum())),
    ("Writable-folder probe variants", 60, cmds[cmds["cmd"].map(is_marker_probe)]["cmd"].nunique()),
    ("Sessions with the plainer writable-folder probes", 2,
     int(uses(sessions, lambda c: WRITABLE.match(c) is not None and not is_marker_probe(c)).sum())),
    ("echo SHELL_TEST-only sessions", 58, len(shell_test_only)),
    ("echo SHELL_TEST-only sessions: IPs", 3, shell_test_only["ip_id"].nunique()),
    ("echo SHELL_TEST-only sessions: days", "Nov 3, Nov 4, Nov 8", days_of(shell_test_only["time"])),
    ("Rival botnets named in the kill list", 10,
     len(next(RIVALS.search(c) for c in cmds[cmds["remove_rivals"]]["cmd"] if RIVALS.search(c))[1].split())),
    ("Hex-echoed kill scripts", 1, len(rival_scripts)),
    ("Hex-echoed kill script: echo lines", 3, int(rivals_hex["commands"].map(lambda cs: sum("echo -ne" in c for c in cs)).sum())),
    ("Hex-echoed kill script: botnet names", 20, len([k for k in killed if k not in CPU_BUILDS])),
    ("Hex-echoed kill script: CPU builds", 6, len([k for k in killed if k in CPU_BUILDS])),
    ("Hex-echoed kill script: also kills deleted and hidden programs", True, all(p in "".join(rivals_hex["commands"].map(hex_script))
                                                             for p in ('*"(deleted)"*', '*"/."*'))),
    ("Hex-echoed kill script: times the bot ran it", 10, int(rivals_hex["commands"].map(lambda cs: sum(re.fullmatch(r"sh \S+", c) is not None
                                                                              for c in cs)).sum())),
    ("Hex-echoed kill script: holds no address", True, not re.search(r"\d{1,3}(\.\d{1,3}){3}", "".join(rivals_hex["commands"].map(hex_script)))),
    ("Droppers that passed telnet to their payload", 2,
     download_lines[download_lines.str.contains(r"\bsh (?:-s |\S+\.sh )telnet", regex=True)].nunique()),
    ("Every dropper tried several download tools", True, bool((dropper_tools.map(len) >= 2).all())),
    ("Droppers that chained their tools with ||, each running only if the one before failed", "19 of 23",
     f"{int(downloaders['commands'].map(lambda cs: any(CHAINED.search(c.replace(chr(92) * 2, chr(92))) for c in cs)).sum())} of {len(downloaders)}"),
    ("Download ports", "80, 443, 8081, 9087",
     ", ".join(str(p) for p in sorted({p for ps in sessions["url_ports"] for p in ps if p}))),
    ("A dropper also used TFTP", True, bool(download_lines.str.contains("tftp", regex=False).any())),
    ("Download sessions using ftpget", 6, len(ftp)),
    ("Download sessions using ftpget: IPs", 5, ftp["ip_id"].nunique()),
    ("Download sessions using ftpget: first day", "Nov 8", day(ftp["time"].min())),
    ("Download sessions using ftpget: all of them download sessions", True, bool(ftp["downloaded"].all())),
    ("Download sessions using ftpget: none from Nov 3 to Nov 6", True, not uses(early, lambda c: "ftpget" in c).any()),
    ("Random-word BusyBox checks: sessions", 12, len(random_word)),
    ("Random-word BusyBox checks: IPs", 5, random_word["ip_id"].nunique()),
    ("Random-word BusyBox checks: first day", "Nov 9", day(random_word["time"].min())),
    ("Random-word BusyBox checks: each after enable, system, shell, sh", True, bool(random_word["commands"].map(lambda c: c[:4] == menu_opening).all())),
    ("Random-word BusyBox checks: none from Nov 3 to Nov 6", True, not uses(early, RANDOM_WORD.fullmatch).any()),
    ("Random-word BusyBox checks: sessions the first detection 1 missed", 1,
     int(uses(random_word, lambda c: RANDOM_WORD.fullmatch(c) and not FIRST_DETECTION_1.search(c)).sum())),
    ("First NUL-word bot: sessions", 122, len(nul_1)),
    ("First NUL-word bot: IPs", 1, nul_1["ip_id"].nunique()),
    ("First NUL-word bot: days", "Nov 3, Nov 4, Nov 5, Nov 6", days_of(nul_1["time"])),
    ("Second NUL-word bot: sessions", 35, len(nul_2)),
    ("Second NUL-word bot: IPs", 1, nul_2["ip_id"].nunique()),
    ("Second NUL-word bot: days", "Nov 10", days_of(nul_2["time"])),
    ("Second NUL-word bot: every session hit the 30-second limit", True, bool((nul_2["ended"] == "timed_out").all())),
    ("Second NUL-word bot: an IP never seen before Nov 10", True, not set(nul_2["ip_id"]) & set(visits[visits["time"] < utc("2026-11-10 00:00:00")]["ip_id"])),
    ("Top 10 usernames by IPs", "root 351, admin 261, default 84, guest 60, support 57, user 41, telnetadmin 40, "
     "super 37, telecomadmin 34, Alphanetworks 26", ", ".join(f"{u} {n}" for u, n in usernames["ips"].head(10).items())),
    ("root, admin, default and support are among the top usernames", True,
     {"root", "admin", "default", "support"} <= set(usernames.index[:10])),
    ("Lowest daily silent %, through Nov 10", 8, round(min(day_silent))),
    ("Highest daily silent %, through Nov 10", 14, round(max(day_silent))),
    # The telnet-opening comparison, with its own window: Nov 3 to Nov 6.
    ("Telnet comparison, after: visits", 2450, len(after_telnet)),
    ("Telnet comparison, after: silent", 284, len(silent(after_telnet))),
    ("Telnet comparison, after: silent %", 12, pct(len(silent(after_telnet)), len(after_telnet))),
    ("Telnet comparison, after: sent a login", 2132, len(sent_login(after_telnet))),
    ("Telnet comparison, after: sent a login %", 87, pct(len(sent_login(after_telnet)), len(after_telnet))),
    ("Before: visits", 520, len(before)),
    ("Before: silent", 425, len(silent(before))),
    ("Before: silent %", 82, pct(len(silent(before)), len(before))),
    ("Before: retry-pattern IPs", 11, len(retry_before)),
    ("Before without retry IPs: visits", 279, len(before_no_retry)),
    ("Before without retry IPs: silent", 184, len(silent(before_no_retry))),
    ("Before without retry IPs: silent %", 66, pct(len(silent(before_no_retry)), len(before_no_retry))),
    ("Before: sent a login", 81, len(sent_login(before))),
    ("Before: sent a login %", 16, pct(len(sent_login(before)), len(before))),
    ("Before: IP addresses", 117, before["ip_id"].nunique()),
    ("After: first 24 hours visits", 713, len(first_day)),
    ("After: first 24 hours silent", 99, len(silent(first_day))),
    ("After: first 24 hours silent %", 14, pct(len(silent(first_day)), len(first_day))),
    ("After: answered the telnet opening", 1994, int(after_telnet["telnet_answered"].sum())),
    ("After: answered the telnet opening %", 81, pct(int(after_telnet["telnet_answered"].sum()), len(after_telnet))),
    ("After: retry-pattern IPs on the busiest day", 0, max(len(retry_ips(g)) for _, g in telnet_days)),
    ("After: visits from the 11 retry IPs", 0, len(retry_after)),
    ("IPs seen before and after", 7, len(seen_both)),
    ("…their visits before", 109, len(b_both)),
    ("…silent before", 44, len(silent(b_both))),
    ("…silent % before", 40, pct(len(silent(b_both)), len(b_both))),
    ("…their visits after", 434, len(a_both)),
    ("…silent after", 35, len(silent(a_both))),
    ("…silent % after", 8, pct(len(silent(a_both)), len(a_both))),
    ("Lowest daily silent %", 8, round(min(telnet_day_silent))),
    ("Highest daily silent %", 13, round(max(telnet_day_silent))),
]
for key, n_sessions, lines, ips in [
        ("login", 839, 4805, 330), ("menu_escape", 576, 2349, 211), ("router_cli", 196, 588, 58),
        ("ping_injection", 137, 137, 117), ("look_around", 192, 258, 118), ("writable_probe", 198, 221, 60),
        ("busybox_tag", 82, 82, 28), ("echo_marker", 12, 16, 12), ("shell_test", 58, 58, 3),
        ("download", 23, 23, 12), ("remove_rivals", 2, 2, 2), ("clean_up", 6, 6, 5)]:
    checks.append((f"Playbook: {step.loc[key, 'step']}: sessions", n_sessions, int(step.loc[key, "sessions"])))
    checks.append((f"Playbook: {step.loc[key, 'step']}: lines", lines, int(step.loc[key, "lines"])))
    checks.append((f"Playbook: {step.loc[key, 'step']}: IPs", ips, int(step.loc[key, "IPs"])))

# Reports to AbuseIPDB, from the reporter's record.
if reported is None:
    print(f"Not checked: the AbuseIPDB report counts, because {REPORTED_FILE} doesn't exist.")
else:
    report_day = pd.to_datetime(reported["reported_at"]).dt.strftime("%b %d")
    checks += [("Reported to AbuseIPDB", 170, len(reported)),
               ("…on Nov 8", 49, int((report_day == "Nov 08").sum())),
               ("…on Nov 10", 121, int((report_day == "Nov 10").sum()))]

# The report's "Threat intel" section and its fifth defender step, from the lookup results: the
# year checks of every IP in the window (Nov 10 and 11), their second checks (threat_intel_lookup.py
# --older-reports, Nov 11 and 13) and the scanner check (abuse_review.py --scanners, Nov 13).
INTEL_EXPECTED = {
    "IPs checked": 901,
    "Every IP in the report's window was checked": True,
    "First sightings from": "Nov 2",
    "First sightings to": "Nov 10",
    "Checked on": "Nov 10 and Nov 11",
    "Last checked on": "Nov 11",
    "Shell IPs": 330,
    "Shell IPs reported in the past year": 328,
    "Shell IPs reported in the past year %": 99,
    "Shell IPs reported before the honeypot saw them": 316,
    "Shell IPs reported before the honeypot saw them %": 96,
    "Shell IPs reported before the honeypot saw them, at most": 319,
    "Shell IPs reported before the honeypot saw them, at most %": 97,
    "Reported in the past year": 891,
    "Reported in the past year %": 99,
    "No reports in the past year": 10,
    "Every IP in the window had the scanner check": True,
    "Research scanners, at least": 48,
    "Research scanners that got into the shell": 0,
    "Every research scanner was reported before the honeypot saw it": True,
    "Leaving the scanners out moves each share by under a point": True,
    "Reported before the honeypot saw them": 867,
    "Reported before the honeypot saw them %": 96,
    "Reported before the honeypot saw them, at most": 871,
    "Reported before the honeypot saw them, at most %": 97,
    "Median days reported before the honeypot saw them": 28,
    "IPs with more reports than the 100 AbuseIPDB lists": 397,
    "…of them with no listed report before the honeypot saw them": 121,
    "Every IP the lists couldn't settle had the second check": True,
    "…shown reported before by the second check": 117,
    "…still can't be told after it": 4,
    "Second checks on": "Nov 11 and Nov 13",
    "For each still undecided, the second check counted more than the year check": True,
    "Reported before the honeypot saw them, from a listed report": 750,
    "IPs that tried a download": 12,
    "Download IPs reported before they arrived": 12,
    "Download IPs that can't be told": 0,
    "Download IPs with no reports before they arrived": 0,
    "Download IPs shown only by the second check": 5,
    "Probe campaign IPs reported before us": "58 of 58",
    "ping opening IPs reported before us": "97 of 100",
    "HISILICON IPs reported before us": "8 of 8",
    "uname -m campaign IPs": 3,
    "echo SHELL_TEST campaign IPs": 3,
    "NUL-word bot IPs": 1,
    "Second NUL-word bot IPs": 1,
    "Persistent bots came back dozens of times (24+ sessions each)": True,
    "Persistent bot IPs": 8,
    "Persistent bot IPs reported before us": 6,
    "Persistent bot IPs that can't be told": 2,
    "Persistent bot IPs shown only by the second check": 5,
    "Persistent bot IPs scoring 100 now": 7,
    "Lowest persistent bot score now": 65,
    "Fixed-line ISP IPs": 696,
    "Scoring 75 or more": 551,
    "Scoring 75 or more %": 61,
    "Scoring 75 or more now, with no reports before us": 2,
    "Scoring 75 or more now, with no reports before us, at most": 6,
}
if intel_numbers is None or not intel_numbers["Every IP in the report's window was checked"]:
    looked_up = 0 if intel_ips is None else len(intel_ips)
    print(f"Not checked: the {len(INTEL_EXPECTED)} threat-intel numbers, because {looked_up} of the "
          f"{len(window_ips)} IPs in the window have been looked up. Run "
          "python3 analysis/threat_intel_lookup.py --only abuseipdb --send for the rest.")
else:
    assert set(INTEL_EXPECTED) == set(intel_numbers), "section 10 and this check list different numbers"
    checks += [(f"Threat intel: {k}", v, intel_numbers[k]) for k, v in INTEL_EXPECTED.items()]

# Who reported each IP first, from threat_intel_lookup.py --first-reporter (run Nov 10, 21:50 to 21:52 UTC),
# and the second check for the 5 it couldn't tell (--older-reports, Nov 13 01:35 UTC).
FIRST_EXPECTED = {
    "IPs looked up": 170,
    "Us first": 7,
    "Someone else first": 158,
    "Can't tell from the lists": 5,
    "Every one of those had the second check": True,
    "…someone else first, shown by the second check": 4,
    "…still can't tell after it": 1,
    "Someone else first, with the second check": 162,
    "For each still undecided, the second check counted more than the first lookup": True,
    "Can't tell because ours wasn't among the 100 listed": 5,
    "Us first, all reported on Nov 10": True,
}
if first_numbers is not None:
    assert set(FIRST_EXPECTED) == set(first_numbers), "section 10 and this check list different numbers"
if first_numbers is None or not first_numbers["Every one of those had the second check"]:
    print(f"Not checked: the {len(FIRST_EXPECTED)} first-reporter numbers, until every reported IP has been looked "
          "up and each one it couldn't tell about has had the second check (threat_intel_lookup.py --older-reports).")
else:
    checks += [(f"First reporter: {k}", v, first_numbers[k]) for k, v in FIRST_EXPECTED.items()]

report_check = pd.DataFrame(checks, columns=["number in README_REPORT.md", "report", "notebook"])
report_check["match"] = [r == n for r, n in zip(report_check["report"], report_check["notebook"])]
wrong = report_check[~report_check["match"]]
print(f"{report_check['match'].sum()} of {len(report_check)} numbers match the report."
      + ("" if wrong.empty else " These don't:"))
display(wrong if not wrong.empty else report_check)

## Save the charts as images

Each chart's camera button downloads a PNG. To write PNG and SVG files for all of them at once, set
`EXPORT = True` and run everything. They go in `analysis/figures/`, which git ignores, named with
the theme, such as `1-funnel-light.png`. The website needs both themes, so run it once with
`THEME = "light"` and once with `THEME = "dark"`. Copy the files you want into the site or report
on purpose. The charts hold only counts and masked labels.

In [ ]:
EXPORT = False

if EXPORT:
    out = REPO / "analysis" / "figures"
    out.mkdir(exist_ok=True)
    for name, fig in FIGURES.items():
        # The theme is in the name, so a light run and a dark run can sit side by side.
        fig.write_image(out / f"{name}-{THEME}.png", width=1000, scale=2)
        fig.write_image(out / f"{name}-{THEME}.svg", width=1000)
    print(f"Wrote {len(FIGURES) * 2} {THEME} files to {out}")